In [1]:
import pulp
import pandas as pd
from datetime import datetime

# --- 1. AGENT DATABASE SETUP ---
# Update this section with your actual 50 agents
agents_db = {}
for i in range(1, 51):
    if i <= 20: # Morning FT
        agents_db[f"Agent_{i:02d}"] = {"type": "FT", "workdays": [0,1,2,3,4], "window": (6, 9)}
    elif i <= 35: # Afternoon FT
        agents_db[f"Agent_{i:02d}"] = {"type": "FT", "workdays": [1,2,3,4,5], "window": (13, 15)}
    elif i <= 45: # PT Morning
        agents_db[f"Agent_{i:02d}"] = {"type": "PT", "workdays": [0,2,4], "window": (8, 8)}
    else: # Weekend PT
        agents_db[f"Agent_{i:02d}"] = {"type": "PT", "workdays": [5,6], "window": (8, 10)}

def solve_daily_schedule(date_val, demand_dict):
    # Determine weekday (0=Mon, 6=Sun)
    weekday = date_val.weekday()

    # Filter agents available for this day
    avail = [a for a, info in agents_db.items() if weekday in info['workdays']]

    prob = pulp.LpProblem(f"Schedule_{date_val.date()}", pulp.LpMinimize)

    # Variables: Choices for start times
    choices = {}
    
    for a in avail:
        s_min, s_max = agents_db[a]['window']
        for s in range(s_min, s_max + 1):
            choices[(a, s)] = pulp.LpVariable(f"Start_{a}_{s}_{date_val.date()}", cat="Binary")

# Constraints
# 1. Each agent works at most 1 shift per day
    for a in avail:
        s_min, s_max = agents_db[a]['window']
        prob += pulp.lpSum([choices[(a, s)] for s in range(s_min, s_max + 1)]) <= 1

# 2. Coverage vs Brut Demand
for h in range(24):
    coverage = []
    for a in avail:
        duration = 8 if agents_db[a]['type'] == "FT" else 4
        s_min, s_max = agents_db[a]['window']
        for s in range(s_min, s_max + 1):
            if s <= h < s + duration:
# Lunch Logic: 0 FTE during the 4th hour of a FT shift
                if agents_db[a]['type'] == "FT" and h == (s + 4):
                    coverage.append(choices[(a, s)] * 0)
                else:
                    coverage.append(choices[(a, s)] * 1)

# Match the demand for this specific hour
    prob += pulp.lpSum(coverage) >= demand_dict.get(h, 0)

# Solve
prob.solve(pulp.PULP_CBC_CMD(msg=0))

day_results = []
if pulp.LpStatus[prob.status] == 'Optimal':
    for (a, s) in choices:
        if pulp.value(choices[(a, s)]) == 1:
            dur = 8 if agents_db[a]['type'] == "FT" else 4
            day_results.append({
                "Date": date_val.date(),
                "Agent": a,
                "Type": agents_db[a]['type'],
                "Start": f"{s:02d}:00",
                "End": f"{(s+dur)%24:02d}:00"
            })
return day_results

# --- 2. DATA PROCESSING ---

# Load the file
# Expected Columns: A: 'Date', B: 'Day', C: 'Hour', D: 'FTE BRUT'
df = pd.read_excel("need.xlsx")

# Convert Date column to actual datetime objects if they aren't already
df['Date'] = pd.to_datetime(df['Date'])

# Pivot or group the data so we have a dictionary of {Hour: FTE} for every Date
all_schedules = []
unique_dates = df['Date'].unique()

for d in unique_dates:
# Filter data for just this day
    day_data = df[df['Date'] == d]

# Create a dictionary of {Hour: FTE BRUT}
# We use column names or indices: C is index 2, D is index 3
    demand_dict = dict(zip(day_data.iloc[:, 2], day_data.iloc[:, 3]))

    print(f"Calculating schedule for {pd.to_datetime(d).date()}...")

# Run the solver
    day_res = solve_daily_schedule(pd.to_datetime(d), demand_dict)
    all_schedules.extend(day_res)

# --- 3. EXPORT ---
df_final = pd.DataFrame(all_schedules)
df_final.to_excel("final_schedule.xlsx", index=False)
print("Done! Check final_schedule.xlsx")

NameError: name 'avail' is not defined

In [2]:
import pulp
import pandas as pd
from datetime import datetime

# --- 1. AGENT DATABASE SETUP ---
# Update this section with your actual 50 agents
agents_db = {}
for i in range(1, 51):
    if i <= 20: # Morning FT
        agents_db[f"Agent_{i:02d}"] = {"type": "FT", "workdays": [0,1,2,3,4], "window": (6, 9)}
    elif i <= 35: # Afternoon FT
        agents_db[f"Agent_{i:02d}"] = {"type": "FT", "workdays": [1,2,3,4,5], "window": (13, 15)}
    elif i <= 45: # PT Morning
        agents_db[f"Agent_{i:02d}"] = {"type": "PT", "workdays": [0,2,4], "window": (8, 8)}
    else: # Weekend PT
        agents_db[f"Agent_{i:02d}"] = {"type": "PT", "workdays": [5,6], "window": (8, 10)}

def solve_daily_schedule(date_val, demand_dict):
    # Determine weekday (0=Mon, 6=Sun)
    weekday = date_val.weekday()

    # Filter agents available for this day
    avail = [a for a, info in agents_db.items() if weekday in info['workdays']]

    prob = pulp.LpProblem(f"Schedule_{date_val.date()}", pulp.LpMinimize)

    # Variables: Choices for start times
    choices = {}
    
    for a in avail:
        s_min, s_max = agents_db[a]['window']
        for s in range(s_min, s_max + 1):
            choices[(a, s)] = pulp.LpVariable(f"Start_{a}_{s}_{date_val.date()}", cat="Binary")

# Constraints
# 1. Each agent works at most 1 shift per day
    for a in avail:
        s_min, s_max = agents_db[a]['window']
        prob += pulp.lpSum([choices[(a, s)] for s in range(s_min, s_max + 1)]) <= 1

# 2. Coverage vs Brut Demand
for h in range(24):
    coverage = []
    for a in avail:
        duration = 8 if agents_db[a]['type'] == "FT" else 4
        s_min, s_max = agents_db[a]['window']
        for s in range(s_min, s_max + 1):
            if s <= h < s + duration:
# Lunch Logic: 0 FTE during the 4th hour of a FT shift
                if agents_db[a]['type'] == "FT" and h == (s + 4):
                    coverage.append(choices[(a, s)] * 0)
                else:
                    coverage.append(choices[(a, s)] * 1)

# Match the demand for this specific hour
    prob += pulp.lpSum(coverage) >= demand_dict.get(h, 0)

# Solve
prob.solve(pulp.PULP_CBC_CMD(msg=0))

day_results = []
if pulp.LpStatus[prob.status] == 'Optimal':
    for (a, s) in choices:
        if pulp.value(choices[(a, s)]) == 1:
            dur = 8 if agents_db[a]['type'] == "FT" else 4
            day_results.append({
                "Date": date_val.date(),
                "Agent": a,
                "Type": agents_db[a]['type'],
                "Start": f"{s:02d}:00",
                "End": f"{(s+dur)%24:02d}:00"
            })
return day_results

# --- 2. DATA PROCESSING ---

# Load the file
# Expected Columns: A: 'Date', B: 'Day', C: 'Hour', D: 'FTE BRUT'
df = pd.read_excel("need.xlsx")

# Convert Date column to actual datetime objects if they aren't already
df['Date'] = pd.to_datetime(df['Date'])

# Pivot or group the data so we have a dictionary of {Hour: FTE} for every Date
all_schedules = []
unique_dates = df['Date'].unique()

for d in unique_dates:
# Filter data for just this day
    day_data = df[df['Date'] == d]

# Create a dictionary of {Hour: FTE BRUT}
# We use column names or indices: C is index 2, D is index 3
    demand_dict = dict(zip(day_data.iloc[:, 2], day_data.iloc[:, 3]))

    print(f"Calculating schedule for {pd.to_datetime(d).date()}...")

# Run the solver
    day_res = solve_daily_schedule(pd.to_datetime(d), demand_dict)
    all_schedules.extend(day_res)

# --- 3. EXPORT ---
df_final = pd.DataFrame(all_schedules)
df_final.to_excel("final_schedule.xlsx", index=False)
print("Done! Check final_schedule.xlsx")

NameError: name 'avail' is not defined

In [3]:
import pulp
import pandas as pd
from datetime import datetime

# --- 1. AGENT DATABASE SETUP ---
# Update this section with your actual 50 agents
agents_db = {}
for i in range(1, 51):
    if i <= 20: # Morning FT
        agents_db[f"Agent_{i:02d}"] = {"type": "FT", "workdays": [0,1,2,3,4], "window": (6, 9)}
    elif i <= 35: # Afternoon FT
        agents_db[f"Agent_{i:02d}"] = {"type": "FT", "workdays": [1,2,3,4,5], "window": (13, 15)}
    elif i <= 45: # PT Morning
        agents_db[f"Agent_{i:02d}"] = {"type": "PT", "workdays": [0,2,4], "window": (8, 8)}
    else: # Weekend PT
        agents_db[f"Agent_{i:02d}"] = {"type": "PT", "workdays": [5,6], "window": (8, 10)}

def solve_daily_schedule(date_val, demand_dict):
    # Determine weekday (0=Mon, 6=Sun)
    weekday = date_val.weekday()

    # Filter agents available for this day
    avail = [a for a, info in agents_db.items() if weekday in info['workdays']]

    prob = pulp.LpProblem(f"Schedule_{date_val.date()}", pulp.LpMinimize)

    # Variables: Choices for start times
    choices = {}
    
    for a in avail:
        s_min, s_max = agents_db[a]['window']
        for s in range(s_min, s_max + 1):
            choices[(a, s)] = pulp.LpVariable(f"Start_{a}_{s}_{date_val.date()}", cat="Binary")

# Constraints
# 1. Each agent works at most 1 shift per day
    for a in avail:
        s_min, s_max = agents_db[a]['window']
        prob += pulp.lpSum([choices[(a, s)] for s in range(s_min, s_max + 1)]) <= 1

# 2. Coverage vs Brut Demand
    for h in range(24):
        coverage = []
        for a in avail:
            duration = 8 if agents_db[a]['type'] == "FT" else 4
            s_min, s_max = agents_db[a]['window']
            for s in range(s_min, s_max + 1):
                if s <= h < s + duration:
    # Lunch Logic: 0 FTE during the 4th hour of a FT shift
                    if agents_db[a]['type'] == "FT" and h == (s + 4):
                        coverage.append(choices[(a, s)] * 0)
                    else:
                        coverage.append(choices[(a, s)] * 1)

# Match the demand for this specific hour
    prob += pulp.lpSum(coverage) >= demand_dict.get(h, 0)

# Solve
prob.solve(pulp.PULP_CBC_CMD(msg=0))

day_results = []
if pulp.LpStatus[prob.status] == 'Optimal':
    for (a, s) in choices:
        if pulp.value(choices[(a, s)]) == 1:
            dur = 8 if agents_db[a]['type'] == "FT" else 4
            day_results.append({
                "Date": date_val.date(),
                "Agent": a,
                "Type": agents_db[a]['type'],
                "Start": f"{s:02d}:00",
                "End": f"{(s+dur)%24:02d}:00"
            })
return day_results

# --- 2. DATA PROCESSING ---

# Load the file
# Expected Columns: A: 'Date', B: 'Day', C: 'Hour', D: 'FTE BRUT'
df = pd.read_excel("need.xlsx")

# Convert Date column to actual datetime objects if they aren't already
df['Date'] = pd.to_datetime(df['Date'])

# Pivot or group the data so we have a dictionary of {Hour: FTE} for every Date
all_schedules = []
unique_dates = df['Date'].unique()

for d in unique_dates:
# Filter data for just this day
    day_data = df[df['Date'] == d]

# Create a dictionary of {Hour: FTE BRUT}
# We use column names or indices: C is index 2, D is index 3
    demand_dict = dict(zip(day_data.iloc[:, 2], day_data.iloc[:, 3]))

    print(f"Calculating schedule for {pd.to_datetime(d).date()}...")

# Run the solver
    day_res = solve_daily_schedule(pd.to_datetime(d), demand_dict)
    all_schedules.extend(day_res)

# --- 3. EXPORT ---
df_final = pd.DataFrame(all_schedules)
df_final.to_excel("final_schedule.xlsx", index=False)
print("Done! Check final_schedule.xlsx")

NameError: name 'prob' is not defined

In [4]:
import pulp
import pandas as pd
from datetime import datetime

# --- 1. AGENT DATABASE SETUP ---
# Update this section with your actual 50 agents
agents_db = {}
for i in range(1, 51):
    if i <= 20: # Morning FT
        agents_db[f"Agent_{i:02d}"] = {"type": "FT", "workdays": [0,1,2,3,4], "window": (6, 9)}
    elif i <= 35: # Afternoon FT
        agents_db[f"Agent_{i:02d}"] = {"type": "FT", "workdays": [1,2,3,4,5], "window": (13, 15)}
    elif i <= 45: # PT Morning
        agents_db[f"Agent_{i:02d}"] = {"type": "PT", "workdays": [0,2,4], "window": (8, 8)}
    else: # Weekend PT
        agents_db[f"Agent_{i:02d}"] = {"type": "PT", "workdays": [5,6], "window": (8, 10)}

def solve_daily_schedule(date_val, demand_dict):
    # Determine weekday (0=Mon, 6=Sun)
    weekday = date_val.weekday()

    # Filter agents available for this day
    avail = [a for a, info in agents_db.items() if weekday in info['workdays']]

    prob = pulp.LpProblem(f"Schedule_{date_val.date()}", pulp.LpMinimize)

    # Variables: Choices for start times
    choices = {}
    
    for a in avail:
        s_min, s_max = agents_db[a]['window']
        for s in range(s_min, s_max + 1):
            choices[(a, s)] = pulp.LpVariable(f"Start_{a}_{s}_{date_val.date()}", cat="Binary")

# Constraints
# 1. Each agent works at most 1 shift per day
    for a in avail:
        s_min, s_max = agents_db[a]['window']
        prob += pulp.lpSum([choices[(a, s)] for s in range(s_min, s_max + 1)]) <= 1

# 2. Coverage vs Brut Demand
    for h in range(24):
        coverage = []
        for a in avail:
            duration = 8 if agents_db[a]['type'] == "FT" else 4
            s_min, s_max = agents_db[a]['window']
            for s in range(s_min, s_max + 1):
                if s <= h < s + duration:
    # Lunch Logic: 0 FTE during the 4th hour of a FT shift
                    if agents_db[a]['type'] == "FT" and h == (s + 4):
                        coverage.append(choices[(a, s)] * 0)
                    else:
                        coverage.append(choices[(a, s)] * 1)

# Match the demand for this specific hour
    prob += pulp.lpSum(coverage) >= demand_dict.get(h, 0)

# Solve
    prob.solve(pulp.PULP_CBC_CMD(msg=0))

day_results = []
if pulp.LpStatus[prob.status] == 'Optimal':
    for (a, s) in choices:
        if pulp.value(choices[(a, s)]) == 1:
            dur = 8 if agents_db[a]['type'] == "FT" else 4
            day_results.append({
                "Date": date_val.date(),
                "Agent": a,
                "Type": agents_db[a]['type'],
                "Start": f"{s:02d}:00",
                "End": f"{(s+dur)%24:02d}:00"
            })
return day_results

# --- 2. DATA PROCESSING ---

# Load the file
# Expected Columns: A: 'Date', B: 'Day', C: 'Hour', D: 'FTE BRUT'
df = pd.read_excel("need.xlsx")

# Convert Date column to actual datetime objects if they aren't already
df['Date'] = pd.to_datetime(df['Date'])

# Pivot or group the data so we have a dictionary of {Hour: FTE} for every Date
all_schedules = []
unique_dates = df['Date'].unique()

for d in unique_dates:
# Filter data for just this day
    day_data = df[df['Date'] == d]

# Create a dictionary of {Hour: FTE BRUT}
# We use column names or indices: C is index 2, D is index 3
    demand_dict = dict(zip(day_data.iloc[:, 2], day_data.iloc[:, 3]))

    print(f"Calculating schedule for {pd.to_datetime(d).date()}...")

# Run the solver
    day_res = solve_daily_schedule(pd.to_datetime(d), demand_dict)
    all_schedules.extend(day_res)

# --- 3. EXPORT ---
df_final = pd.DataFrame(all_schedules)
df_final.to_excel("final_schedule.xlsx", index=False)
print("Done! Check final_schedule.xlsx")

NameError: name 'prob' is not defined

In [5]:
import pulp
import pandas as pd
from datetime import datetime

# --- 1. AGENT DATABASE SETUP ---
# Update this section with your actual 50 agents
agents_db = {}
for i in range(1, 51):
    if i <= 20: # Morning FT
        agents_db[f"Agent_{i:02d}"] = {"type": "FT", "workdays": [0,1,2,3,4], "window": (6, 9)}
    elif i <= 35: # Afternoon FT
        agents_db[f"Agent_{i:02d}"] = {"type": "FT", "workdays": [1,2,3,4,5], "window": (13, 15)}
    elif i <= 45: # PT Morning
        agents_db[f"Agent_{i:02d}"] = {"type": "PT", "workdays": [0,2,4], "window": (8, 8)}
    else: # Weekend PT
        agents_db[f"Agent_{i:02d}"] = {"type": "PT", "workdays": [5,6], "window": (8, 10)}

def solve_daily_schedule(date_val, demand_dict):
    # Determine weekday (0=Mon, 6=Sun)
    weekday = date_val.weekday()

    # Filter agents available for this day
    avail = [a for a, info in agents_db.items() if weekday in info['workdays']]

    prob = pulp.LpProblem(f"Schedule_{date_val.date()}", pulp.LpMinimize)

    # Variables: Choices for start times
    choices = {}
    
    for a in avail:
        s_min, s_max = agents_db[a]['window']
        for s in range(s_min, s_max + 1):
            choices[(a, s)] = pulp.LpVariable(f"Start_{a}_{s}_{date_val.date()}", cat="Binary")

# Constraints
# 1. Each agent works at most 1 shift per day
    for a in avail:
        s_min, s_max = agents_db[a]['window']
        prob += pulp.lpSum([choices[(a, s)] for s in range(s_min, s_max + 1)]) <= 1

# 2. Coverage vs Brut Demand
    for h in range(24):
        coverage = []
        for a in avail:
            duration = 8 if agents_db[a]['type'] == "FT" else 4
            s_min, s_max = agents_db[a]['window']
            for s in range(s_min, s_max + 1):
                if s <= h < s + duration:
    # Lunch Logic: 0 FTE during the 4th hour of a FT shift
                    if agents_db[a]['type'] == "FT" and h == (s + 4):
                        coverage.append(choices[(a, s)] * 0)
                    else:
                        coverage.append(choices[(a, s)] * 1)

# Match the demand for this specific hour
    prob += pulp.lpSum(coverage) >= demand_dict.get(h, 0)

# Solve
    prob.solve(pulp.PULP_CBC_CMD(msg=0))

day_results = []
    if pulp.LpStatus[prob.status] == 'Optimal':
        for (a, s) in choices:
            if pulp.value(choices[(a, s)]) == 1:
                dur = 8 if agents_db[a]['type'] == "FT" else 4
                day_results.append({
                    "Date": date_val.date(),
                    "Agent": a,
                    "Type": agents_db[a]['type'],
                    "Start": f"{s:02d}:00",
                    "End": f"{(s+dur)%24:02d}:00"
                })
    return day_results

# --- 2. DATA PROCESSING ---

# Load the file
# Expected Columns: A: 'Date', B: 'Day', C: 'Hour', D: 'FTE BRUT'
df = pd.read_excel("need.xlsx")

# Convert Date column to actual datetime objects if they aren't already
df['Date'] = pd.to_datetime(df['Date'])

# Pivot or group the data so we have a dictionary of {Hour: FTE} for every Date
all_schedules = []
unique_dates = df['Date'].unique()

for d in unique_dates:
# Filter data for just this day
    day_data = df[df['Date'] == d]

# Create a dictionary of {Hour: FTE BRUT}
# We use column names or indices: C is index 2, D is index 3
    demand_dict = dict(zip(day_data.iloc[:, 2], day_data.iloc[:, 3]))

    print(f"Calculating schedule for {pd.to_datetime(d).date()}...")

# Run the solver
    day_res = solve_daily_schedule(pd.to_datetime(d), demand_dict)
    all_schedules.extend(day_res)

# --- 3. EXPORT ---
df_final = pd.DataFrame(all_schedules)
df_final.to_excel("final_schedule.xlsx", index=False)
print("Done! Check final_schedule.xlsx")

IndentationError: unexpected indent (2397608187.py, line 62)

In [6]:
import pulp
import pandas as pd
from datetime import datetime

# --- 1. AGENT DATABASE SETUP ---
# Update this section with your actual 50 agents
agents_db = {}
for i in range(1, 51):
    if i <= 20: # Morning FT
        agents_db[f"Agent_{i:02d}"] = {"type": "FT", "workdays": [0,1,2,3,4], "window": (6, 9)}
    elif i <= 35: # Afternoon FT
        agents_db[f"Agent_{i:02d}"] = {"type": "FT", "workdays": [1,2,3,4,5], "window": (13, 15)}
    elif i <= 45: # PT Morning
        agents_db[f"Agent_{i:02d}"] = {"type": "PT", "workdays": [0,2,4], "window": (8, 8)}
    else: # Weekend PT
        agents_db[f"Agent_{i:02d}"] = {"type": "PT", "workdays": [5,6], "window": (8, 10)}

def solve_daily_schedule(date_val, demand_dict):
    # Determine weekday (0=Mon, 6=Sun)
    weekday = date_val.weekday()

    # Filter agents available for this day
    avail = [a for a, info in agents_db.items() if weekday in info['workdays']]

    prob = pulp.LpProblem(f"Schedule_{date_val.date()}", pulp.LpMinimize)

    # Variables: Choices for start times
    choices = {}
    
    for a in avail:
        s_min, s_max = agents_db[a]['window']
        for s in range(s_min, s_max + 1):
            choices[(a, s)] = pulp.LpVariable(f"Start_{a}_{s}_{date_val.date()}", cat="Binary")

# Constraints
# 1. Each agent works at most 1 shift per day
    for a in avail:
        s_min, s_max = agents_db[a]['window']
        prob += pulp.lpSum([choices[(a, s)] for s in range(s_min, s_max + 1)]) <= 1

# 2. Coverage vs Brut Demand
    for h in range(24):
        coverage = []
        for a in avail:
            duration = 8 if agents_db[a]['type'] == "FT" else 4
            s_min, s_max = agents_db[a]['window']
            for s in range(s_min, s_max + 1):
                if s <= h < s + duration:
    # Lunch Logic: 0 FTE during the 4th hour of a FT shift
                    if agents_db[a]['type'] == "FT" and h == (s + 4):
                        coverage.append(choices[(a, s)] * 0)
                    else:
                        coverage.append(choices[(a, s)] * 1)

# Match the demand for this specific hour
    prob += pulp.lpSum(coverage) >= demand_dict.get(h, 0)

# Solve
    prob.solve(pulp.PULP_CBC_CMD(msg=0))

    day_results = []
    if pulp.LpStatus[prob.status] == 'Optimal':
        for (a, s) in choices:
            if pulp.value(choices[(a, s)]) == 1:
                dur = 8 if agents_db[a]['type'] == "FT" else 4
                day_results.append({
                    "Date": date_val.date(),
                    "Agent": a,
                    "Type": agents_db[a]['type'],
                    "Start": f"{s:02d}:00",
                    "End": f"{(s+dur)%24:02d}:00"
                })
    return day_results

# --- 2. DATA PROCESSING ---

# Load the file
# Expected Columns: A: 'Date', B: 'Day', C: 'Hour', D: 'FTE BRUT'
df = pd.read_excel("need.xlsx")

# Convert Date column to actual datetime objects if they aren't already
df['Date'] = pd.to_datetime(df['Date'])

# Pivot or group the data so we have a dictionary of {Hour: FTE} for every Date
all_schedules = []
unique_dates = df['Date'].unique()

for d in unique_dates:
# Filter data for just this day
    day_data = df[df['Date'] == d]

# Create a dictionary of {Hour: FTE BRUT}
# We use column names or indices: C is index 2, D is index 3
    demand_dict = dict(zip(day_data.iloc[:, 2], day_data.iloc[:, 3]))

    print(f"Calculating schedule for {pd.to_datetime(d).date()}...")

# Run the solver
    day_res = solve_daily_schedule(pd.to_datetime(d), demand_dict)
    all_schedules.extend(day_res)

# --- 3. EXPORT ---
df_final = pd.DataFrame(all_schedules)
df_final.to_excel("final_schedule.xlsx", index=False)
print("Done! Check final_schedule.xlsx")

Calculating schedule for 2026-05-01...
Calculating schedule for 2026-05-02...
Calculating schedule for 2026-05-03...
Calculating schedule for 2026-05-04...
Calculating schedule for 2026-05-05...
Calculating schedule for 2026-05-06...
Calculating schedule for 2026-05-07...
Calculating schedule for 2026-05-08...
Calculating schedule for 2026-05-09...
Calculating schedule for 2026-05-10...
Calculating schedule for 2026-05-11...
Calculating schedule for 2026-05-12...
Calculating schedule for 2026-05-13...
Calculating schedule for 2026-05-14...
Calculating schedule for 2026-05-15...
Calculating schedule for 2026-05-16...
Calculating schedule for 2026-05-17...
Calculating schedule for 2026-05-18...
Calculating schedule for 2026-05-19...
Calculating schedule for 2026-05-20...
Calculating schedule for 2026-05-21...
Calculating schedule for 2026-05-22...
Calculating schedule for 2026-05-23...
Calculating schedule for 2026-05-24...
Calculating schedule for 2026-05-25...
Calculating schedule for 

In [7]:
import pulp
import pandas as pd
from datetime import datetime

# --- CONFIGURATION: YOUR AGENT LIST ---
# This tells the AI who exists and when they can start.
# 0=Mon, 1=Tue, 2=Wed, 3=Thu, 4=Fri, 5=Sat, 6=Sun
agents_db = {}
for i in range(1, 51):
    if i <= 20: # Agents 1-20: Morning FT (Mon-Fri)
        agents_db[f"Agent_{i:02d}"] = {"type": "FT", "workdays": [0,1,2,3,4], "window": (6, 9)}
    elif i <= 35: # Agents 21-35: Afternoon FT (Mon-Fri)
        agents_db[f"Agent_{i:02d}"] = {"type": "FT", "workdays": [0,1,2,3,4], "window": (13, 15)}
    else: # Others: Part Time 
        agents_db[f"Agent_{i:02d}"] = {"type": "PT", "workdays": [0,1,2,3,4,5,6], "window": (8, 10)}

# --- THE ENGINE: DO NOT CHANGE BELOW UNLESS YOU ARE BRAVE ---
def solve_day(date_val, demand_dict):
    weekday = date_val.weekday()
    avail = [a for a, info in agents_db.items() if weekday in info['workdays']]
    prob = pulp.LpProblem(f"Sch_{date_val.date()}", pulp.LpMinimize)
    choices = {}
    for a in avail:
        s_min, s_max = agents_db[a]['window']
        for s in range(s_min, s_max + 1):
            choices[(a, s)] = pulp.LpVariable(f"S_{a}_{s}_{date_val.date()}", cat="Binary")
    for a in avail:
        s_min, s_max = agents_db[a]['window']
        prob += pulp.lpSum([choices[(a, s)] for s in range(s_min, s_max + 1)]) <= 1
    for h in range(24):
        coverage = []
        for a in avail:
            dur = 8 if agents_db[a]['type'] == "FT" else 4
            s_min, s_max = agents_db[a]['window']
            for s in range(s_min, s_max + 1):
                if s <= h < s + dur:
                    if agents_db[a]['type'] == "FT" and h == (s + 4): # Lunch
                        coverage.append(choices[(a, s)] * 0)
                    else: coverage.append(choices[(a, s)] * 1)
        prob += pulp.lpSum(coverage) >= demand_dict.get(h, 0)
    prob.solve(pulp.PULP_CBC_CMD(msg=0))
    res = []
    if pulp.LpStatus[prob.status] == 'Optimal':
        for (a, s) in choices:
            if pulp.value(choices[(a, s)]) == 1:
                dur = 8 if agents_db[a]['type'] == "FT" else 4
                res.append({"Date": date_val.date(), "Agent": a, "Start": f"{s:02d}:00", "End": f"{(s+dur)%24:02d}:00"})
    return res

# --- LOADING YOUR FILE ---
print("Reading need.xlsx...")
df = pd.read_excel("need.xlsx")
df['Date'] = pd.to_datetime(df['Date'])
all_res = []
for d in df['Date'].unique():
    day_data = df[df['Date'] == d]
# We assume Column C is Hour and Column D is FTE BRUT
    demand_dict = dict(zip(day_data.iloc[:, 2], day_data.iloc[:, 3]))
    print(f"Solving for {pd.to_datetime(d).date()}...")
    all_res.extend(solve_day(pd.to_datetime(d), demand_dict))

# --- EXPORTING ---
pd.DataFrame(all_res).to_excel("final_schedule.xlsx", index=False)
print("SUCCESS! Your file 'final_schedule.xlsx' is in your folder.")


Reading need.xlsx...
Solving for 2026-05-01...
Solving for 2026-05-02...
Solving for 2026-05-03...
Solving for 2026-05-04...
Solving for 2026-05-05...
Solving for 2026-05-06...
Solving for 2026-05-07...
Solving for 2026-05-08...
Solving for 2026-05-09...
Solving for 2026-05-10...
Solving for 2026-05-11...
Solving for 2026-05-12...
Solving for 2026-05-13...
Solving for 2026-05-14...
Solving for 2026-05-15...
Solving for 2026-05-16...
Solving for 2026-05-17...
Solving for 2026-05-18...
Solving for 2026-05-19...
Solving for 2026-05-20...
Solving for 2026-05-21...
Solving for 2026-05-22...
Solving for 2026-05-23...
Solving for 2026-05-24...
Solving for 2026-05-25...
Solving for 2026-05-26...
Solving for 2026-05-27...
Solving for 2026-05-28...
Solving for 2026-05-29...
Solving for 2026-05-30...
Solving for 2026-05-31...
Solving for 2026-06-01...
Solving for 2026-06-02...
Solving for 2026-06-03...
Solving for 2026-06-04...
Solving for 2026-06-05...
Solving for 2026-06-06...
Solving for 2026-

In [8]:
import pulp
import pandas as pd
from datetime import datetime

# --- 1. CONFIGURATION: YOUR AGENT LIST --- # Check these workdays! 0=Mon, 4=Fri, 5=Sat, 6=Sun.
agents_db = {}
for i in range(1, 51):
    if i <= 20: # 1-20: Morning (Mon-Fri)
        agents_db[f"Agent_{i:02d}"] = {"type": "FT", "workdays": [0,1,2,3,4], "window": (6, 9)}
    elif i <= 35: # 21-35: Afternoon (Mon-Fri)
        agents_db[f"Agent_{i:02d}"] = {"type": "FT", "workdays": [0,1,2,3,4], "window": (13, 15)}
    else: # 36-50: Everyone else (Full Week)
        agents_db[f"Agent_{i:02d}"] = {"type": "PT", "workdays": [0,1,2,3,4,5,6], "window": (8, 10)}

def solve_day(date_val, demand_dict):
    weekday = date_val.weekday()
    avail = [a for a, info in agents_db.items() if weekday in info['workdays']]
    
    # DEBUG: See how many agents are available
    print(f"--- Day: {date_val.date()} (Weekday {weekday}) | Agents Available: {len(avail)} ---")
    
    if len(avail) == 0:
        print("!!! WARNING: No agents available for this day of the week.")
        return []

    prob = pulp.LpProblem(f"Sch_{date_val.date()}", pulp.LpMinimize)
    choices = {}
    for a in avail:
        s_min, s_max = agents_db[a]['window']
        for s in range(s_min, s_max + 1):
            choices[(a, s)] = pulp.LpVariable(f"S_{a}_{s}_{date_val.date()}", cat="Binary")

    # Constraint: Max 1 shift
    for a in avail:
        s_min, s_max = agents_db[a]['window']
        prob += pulp.lpSum([choices[(a, s)] for s in range(s_min, s_max + 1)]) <= 1

    # Constraint: Coverage with "Slack" (allows it to finish even if understaffed)
    for h in range(24):
        coverage = []
        for a in avail:
            dur = 8 if agents_db[a]['type'] == "FT" else 4
            s_min, s_max = agents_db[a]['window']
            for s in range(s_min, s_max + 1):
                if s <= h < s + dur:
                    if agents_db[a]['type'] == "FT" and h == (s + 4): # Lunch
                        coverage.append(choices[(a, s)] * 0)
                    else: coverage.append(choices[(a, s)] * 1)
        
        # This 'slack' variable makes the code 'Optimal' even if you don't have enough staff
        slack = pulp.LpVariable(f"Slack_{h}", lowBound=0)
        prob += pulp.lpSum(coverage) + slack >= demand_dict.get(h, 0)

    # Goal: Minimize agents used (you can change this to minimize overstaffing)
    prob += pulp.lpSum([choices[v] for v in choices])
    
    prob.solve(pulp.PULP_CBC_CMD(msg=0))
    
    res = []
    print(f"Status: {pulp.LpStatus[prob.status]}") # Should say 'Optimal'
    
    for (a, s) in choices:
        if pulp.value(choices[(a, s)]) == 1:
            dur = 8 if agents_db[a]['type'] == "FT" else 4
            res.append({"Date": date_val.date(), "Agent": a, "Start": f"{s:02d}:00", "End": f"{(s+dur)%24:02d}:00"})
    
    print(f"Assignments made: {len(res)}")
    return res

# --- LOADING AND RUNNING ---
try:
    df = pd.read_excel("need.xlsx")
    print(f"Successfully loaded need.xlsx with {len(df)} rows.")
    
    # If Column names are exactly as you described:
    # A: Date, B: Day, C: Hour, D: FTE BRUT
    # Let's clean the column names to be safe
    df.columns = ['Date', 'Day', 'Hour', 'FTE_Brut']
    df['Date'] = pd.to_datetime(df['Date'])
    
    all_res = []
    for d in df['Date'].unique():
        day_data = df[df['Date'] == d]
        # Map Hour to FTE
        demand_dict = dict(zip(day_data['Hour'], day_data['FTE_Brut']))
        all_res.extend(solve_day(pd.to_datetime(d), demand_dict))

    if len(all_res) > 0:
        output_df = pd.DataFrame(all_res)
        output_df.to_excel("final_schedule.xlsx", index=False)
        print("\n*** SUCCESS! Check final_schedule.xlsx ***")
    else:
        print("\n*** ERROR: No assignments were generated. Check your agent availability. ***")

except Exception as e:
    print(f"\n*** ERROR: {e} ***")

Successfully loaded need.xlsx with 2208 rows.

*** ERROR: Length mismatch: Expected axis has 14 elements, new values have 4 elements ***


In [9]:
# --- UPDATED LOADING AND RUNNING SECTION ---
try:
    # 1. Load the file
    df = pd.read_excel("need.xlsx")
    
    # 2. Let's see what the computer actually sees (Diagnostic)
    actual_col_count = len(df.columns)
    print(f"I found {actual_col_count} columns in your Excel file.")
    print(f"Column names found: {list(df.columns)}")

    # 3. Instead of renaming, we just pick by position:
    # We assume: Col 0 = Date, Col 2 = Hour, Col 3 = FTE_Brut
    # (We skip index 1 because that's usually the 'Day' string)
    
    # Create a cleaned version of the data
    clean_df = pd.DataFrame()
    clean_df['Date'] = pd.to_datetime(df.iloc[:, 0]) # Column A
    clean_df['Hour'] = df.iloc[:, 2]                 # Column C
    clean_df['FTE_Brut'] = df.iloc[:, 4]             # Column E
    
    print(f"Successfully processed {len(clean_df)} rows of data.")

    all_res = []
    for d in clean_df['Date'].unique():
        # Convert numpy datetime back to pandas timestamp for the solver
        ts = pd.to_datetime(d)
        day_data = clean_df[clean_df['Date'] == d]
        
        # Map Hour to FTE
        demand_dict = dict(zip(day_data['Hour'], day_data['FTE_Brut']))
        
        # Solve for the day
        all_res.extend(solve_day(ts, demand_dict))

    if len(all_res) > 0:
        output_df = pd.DataFrame(all_res)
        output_df.to_excel("final_schedule.xlsx", index=False)
        print("\n*** SUCCESS! Check final_schedule.xlsx ***")
    else:
        print("\n*** ERROR: No assignments were generated. Check the text above for warnings. ***")

except Exception as e:
    # This will tell us EXACTLY what went wrong
    import traceback
    print("\n*** CRITICAL ERROR DETAILS ***")
    print(traceback.format_exc())

I found 14 columns in your Excel file.
Column names found: ['Date', 'Day', 'Hour', 'Volume', 'Sizing FTEs', 'Calls/Day/FTE', 'Cases/Day/FTE', 'Service Level %', 'Avg Wait Time (s)', 'Occupancy %', 'P(Wait) %', 'Immediate Answer %', 'Shift', 'Weekend']
Successfully processed 2208 rows of data.
--- Day: 2026-05-01 (Weekday 4) | Agents Available: 50 ---
Status: Optimal
Assignments made: 0
--- Day: 2026-05-02 (Weekday 5) | Agents Available: 15 ---
Status: Optimal
Assignments made: 0
--- Day: 2026-05-03 (Weekday 6) | Agents Available: 15 ---
Status: Optimal
Assignments made: 0
--- Day: 2026-05-04 (Weekday 0) | Agents Available: 50 ---
Status: Optimal
Assignments made: 0
--- Day: 2026-05-05 (Weekday 1) | Agents Available: 50 ---
Status: Optimal
Assignments made: 0
--- Day: 2026-05-06 (Weekday 2) | Agents Available: 50 ---
Status: Optimal
Assignments made: 0
--- Day: 2026-05-07 (Weekday 3) | Agents Available: 50 ---
Status: Optimal
Assignments made: 0
--- Day: 2026-05-08 (Weekday 4) | Agents 

Status: Optimal
Assignments made: 0
--- Day: 2026-07-24 (Weekday 4) | Agents Available: 50 ---
Status: Optimal
Assignments made: 0
--- Day: 2026-07-25 (Weekday 5) | Agents Available: 15 ---
Status: Optimal
Assignments made: 0
--- Day: 2026-07-26 (Weekday 6) | Agents Available: 15 ---
Status: Optimal
Assignments made: 0
--- Day: 2026-07-27 (Weekday 0) | Agents Available: 50 ---
Status: Optimal
Assignments made: 0
--- Day: 2026-07-28 (Weekday 1) | Agents Available: 50 ---
Status: Optimal
Assignments made: 0
--- Day: 2026-07-29 (Weekday 2) | Agents Available: 50 ---
Status: Optimal
Assignments made: 0
--- Day: 2026-07-30 (Weekday 3) | Agents Available: 50 ---
Status: Optimal
Assignments made: 0
--- Day: 2026-07-31 (Weekday 4) | Agents Available: 50 ---
Status: Optimal
Assignments made: 0

*** ERROR: No assignments were generated. Check the text above for warnings. ***


In [10]:
# --- UPDATED LOADING AND RUNNING SECTION ---
try:
    # 1. Load the file
    df = pd.read_excel("need.xlsx")
    
    # 2. Let's see what the computer actually sees (Diagnostic)
    actual_col_count = len(df.columns)
    print(f"I found {actual_col_count} columns in your Excel file.")
    print(f"Column names found: {list(df.columns)}")

    # 3. Instead of renaming, we just pick by position:
    # We assume: Col 0 = Date, Col 2 = Hour, Col 3 = FTE_Brut
    # (We skip index 1 because that's usually the 'Day' string)
    
    # Create a cleaned version of the data
    clean_df = pd.DataFrame()
    clean_df['Date'] = pd.to_datetime(df.iloc[:, 0]) # Column A
    clean_df['Hour'] = df.iloc[:, 2]                 # Column C
    clean_df['FTE_Brut'] = df.iloc[:, 4]             # Column E
    
    print(f"Successfully processed {len(clean_df)} rows of data.")

    all_res = []
    for d in clean_df['Date'].unique():
        # Convert numpy datetime back to pandas timestamp for the solver
        ts = pd.to_datetime(d)
        day_data = clean_df[clean_df['Date'] == d]
        
        # Map Hour to FTE
        demand_dict = dict(zip(day_data['Hour'], day_data['FTE_Brut']))
        
        # Solve for the day
        all_res.extend(solve_day(ts, demand_dict))

    if len(all_res) > 0:
        output_df = pd.DataFrame(all_res)
        output_df.to_excel("final_schedule.xlsx", index=False)
        print("\n*** SUCCESS! Check final_schedule.xlsx ***")
    else:
        print("\n*** ERROR: No assignments were generated. Check the text above for warnings. ***")

except Exception as e:
    # This will tell us EXACTLY what went wrong
    import traceback
    print("\n*** CRITICAL ERROR DETAILS ***")
    print(traceback.format_exc())


*** CRITICAL ERROR DETAILS ***
Traceback (most recent call last):
  File "C:\Users\B622UD\AppData\Local\Temp\ipykernel_8708\1142978967.py", line 4, in <module>
    df = pd.read_excel("need.xlsx")
         ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\B622UD\AppData\Local\anaconda3\Lib\site-packages\pandas\io\excel\_base.py", line 478, in read_excel
    io = ExcelFile(io, storage_options=storage_options, engine=engine)
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\B622UD\AppData\Local\anaconda3\Lib\site-packages\pandas\io\excel\_base.py", line 1496, in __init__
    ext = inspect_excel_format(
          ^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\B622UD\AppData\Local\anaconda3\Lib\site-packages\pandas\io\excel\_base.py", line 1371, in inspect_excel_format
    with get_handle(
         ^^^^^^^^^^^
  File "C:\Users\B622UD\AppData\Local\anaconda3\Lib\site-packages\pandas\io\common.py", line 868, in get_handle
    handle = open(handle, ioargs.mode)
       

In [11]:
# --- UPDATED LOADING AND RUNNING SECTION ---
try:
    # 1. Load the file
    df = pd.read_excel("need.xlsx")
    
    # 2. Let's see what the computer actually sees (Diagnostic)
    actual_col_count = len(df.columns)
    print(f"I found {actual_col_count} columns in your Excel file.")
    print(f"Column names found: {list(df.columns)}")

    # 3. Instead of renaming, we just pick by position:
    # We assume: Col 0 = Date, Col 2 = Hour, Col 3 = FTE_Brut
    # (We skip index 1 because that's usually the 'Day' string)
    
    # Create a cleaned version of the data
    clean_df = pd.DataFrame()
    clean_df['Date'] = pd.to_datetime(df.iloc[:, 0]) # Column A
    clean_df['Hour'] = df.iloc[:, 2]                 # Column C
    clean_df['FTE_Brut'] = df.iloc[:, 4]             # Column E
    
    print(f"Successfully processed {len(clean_df)} rows of data.")

    all_res = []
    for d in clean_df['Date'].unique():
        # Convert numpy datetime back to pandas timestamp for the solver
        ts = pd.to_datetime(d)
        day_data = clean_df[clean_df['Date'] == d]
        
        # Map Hour to FTE
        demand_dict = dict(zip(day_data['Hour'], day_data['FTE_Brut']))
        
        # Solve for the day
        all_res.extend(solve_day(ts, demand_dict))

    if len(all_res) > 0:
        output_df = pd.DataFrame(all_res)
        output_df.to_excel("final_schedule.xlsx", index=False)
        print("\n*** SUCCESS! Check final_schedule.xlsx ***")
    else:
        print("\n*** ERROR: No assignments were generated. Check the text above for warnings. ***")

except Exception as e:
    # This will tell us EXACTLY what went wrong
    import traceback
    print("\n*** CRITICAL ERROR DETAILS ***")
    print(traceback.format_exc())

I found 14 columns in your Excel file.
Column names found: ['Date', 'Day', 'Hour', 'Volume', 'Sizing FTEs', 'Calls/Day/FTE', 'Cases/Day/FTE', 'Service Level %', 'Avg Wait Time (s)', 'Occupancy %', 'P(Wait) %', 'Immediate Answer %', 'Shift', 'Weekend']
Successfully processed 2208 rows of data.
--- Day: 2026-05-01 (Weekday 4) | Agents Available: 50 ---
Status: Optimal
Assignments made: 0
--- Day: 2026-05-02 (Weekday 5) | Agents Available: 15 ---
Status: Optimal
Assignments made: 0
--- Day: 2026-05-03 (Weekday 6) | Agents Available: 15 ---
Status: Optimal
Assignments made: 0
--- Day: 2026-05-04 (Weekday 0) | Agents Available: 50 ---
Status: Optimal
Assignments made: 0
--- Day: 2026-05-05 (Weekday 1) | Agents Available: 50 ---
Status: Optimal
Assignments made: 0
--- Day: 2026-05-06 (Weekday 2) | Agents Available: 50 ---
Status: Optimal
Assignments made: 0
--- Day: 2026-05-07 (Weekday 3) | Agents Available: 50 ---
Status: Optimal
Assignments made: 0
--- Day: 2026-05-08 (Weekday 4) | Agents 

Status: Optimal
Assignments made: 0
--- Day: 2026-07-24 (Weekday 4) | Agents Available: 50 ---
Status: Optimal
Assignments made: 0
--- Day: 2026-07-25 (Weekday 5) | Agents Available: 15 ---
Status: Optimal
Assignments made: 0
--- Day: 2026-07-26 (Weekday 6) | Agents Available: 15 ---
Status: Optimal
Assignments made: 0
--- Day: 2026-07-27 (Weekday 0) | Agents Available: 50 ---
Status: Optimal
Assignments made: 0
--- Day: 2026-07-28 (Weekday 1) | Agents Available: 50 ---
Status: Optimal
Assignments made: 0
--- Day: 2026-07-29 (Weekday 2) | Agents Available: 50 ---
Status: Optimal
Assignments made: 0
--- Day: 2026-07-30 (Weekday 3) | Agents Available: 50 ---
Status: Optimal
Assignments made: 0
--- Day: 2026-07-31 (Weekday 4) | Agents Available: 50 ---
Status: Optimal
Assignments made: 0

*** ERROR: No assignments were generated. Check the text above for warnings. ***


In [12]:
import pulp
import pandas as pd
from datetime import datetime

# --- 1. CONFIGURATION: YOUR AGENT LIST --- # Check these workdays! 0=Mon, 4=Fri, 5=Sat, 6=Sun.
agents_db = {}
for i in range(1, 51):
    if i <= 20: # 1-20: Morning (Mon-Fri)
        agents_db[f"Agent_{i:02d}"] = {"type": "FT", "workdays": [0,1,2,3,4], "window": (6, 9)}
    elif i <= 35: # 21-35: Afternoon (Mon-Fri)
        agents_db[f"Agent_{i:02d}"] = {"type": "FT", "workdays": [0,1,2,3,4], "window": (13, 15)}
    else: # 36-50: Everyone else (Full Week)
        agents_db[f"Agent_{i:02d}"] = {"type": "PT", "workdays": [0,1,2,3,4,5,6], "window": (8, 10)}

def solve_day(date_val, demand_dict):
    weekday = date_val.weekday()
    avail = [a for a, info in agents_db.items() if weekday in info['workdays']]
    
    # DEBUG: See how many agents are available
    print(f"--- Day: {date_val.date()} (Weekday {weekday}) | Agents Available: {len(avail)} ---")
    
    if len(avail) == 0:
        print("!!! WARNING: No agents available for this day of the week.")
        return []

    prob = pulp.LpProblem(f"Sch_{date_val.date()}", pulp.LpMinimize)
    choices = {}
    for a in avail:
        s_min, s_max = agents_db[a]['window']
        for s in range(s_min, s_max + 1):
            choices[(a, s)] = pulp.LpVariable(f"S_{a}_{s}_{date_val.date()}", cat="Binary")

    # Constraint: Max 1 shift
    for a in avail:
        s_min, s_max = agents_db[a]['window']
        prob += pulp.lpSum([choices[(a, s)] for s in range(s_min, s_max + 1)]) <= 1

    # Constraint: Coverage with "Slack" (allows it to finish even if understaffed)
    for h in range(24):
        coverage = []
        for a in avail:
            dur = 8 if agents_db[a]['type'] == "FT" else 4
            s_min, s_max = agents_db[a]['window']
            for s in range(s_min, s_max + 1):
                if s <= h < s + dur:
                    if agents_db[a]['type'] == "FT" and h == (s + 4): # Lunch
                        coverage.append(choices[(a, s)] * 0)
                    else: coverage.append(choices[(a, s)] * 1)
        
        # This 'slack' variable makes the code 'Optimal' even if you don't have enough staff
        slack = pulp.LpVariable(f"Slack_{h}", lowBound=0)
        prob += pulp.lpSum(coverage) + slack >= demand_dict.get(h, 0)

    # Goal: Minimize agents used (you can change this to minimize overstaffing)
    prob += pulp.lpSum([choices[v] for v in choices])
    
    prob.solve(pulp.PULP_CBC_CMD(msg=0))
    
    res = []
    print(f"Status: {pulp.LpStatus[prob.status]}") # Should say 'Optimal'
    
    for (a, s) in choices:
        if pulp.value(choices[(a, s)]) == 1:
            dur = 8 if agents_db[a]['type'] == "FT" else 4
            res.append({"Date": date_val.date(), "Agent": a, "Start": f"{s:02d}:00", "End": f"{(s+dur)%24:02d}:00"})
    
    print(f"Assignments made: {len(res)}")
    return res

# --- UPDATED LOADING AND RUNNING SECTION ---
try:
    # 1. Load the file
    df = pd.read_excel("need.xlsx")
    
    # 2. Let's see what the computer actually sees (Diagnostic)
    actual_col_count = len(df.columns)
    print(f"I found {actual_col_count} columns in your Excel file.")
    print(f"Column names found: {list(df.columns)}")

    # 3. Instead of renaming, we just pick by position:
    # We assume: Col 0 = Date, Col 2 = Hour, Col 3 = FTE_Brut
    # (We skip index 1 because that's usually the 'Day' string)
    
    # Create a cleaned version of the data
    clean_df = pd.DataFrame()
    clean_df['Date'] = pd.to_datetime(df.iloc[:, 0]) # Column A
    clean_df['Hour'] = df.iloc[:, 2]                 # Column C
    clean_df['FTE_Brut'] = df.iloc[:, 4]             # Column E
    
    print(f"Successfully processed {len(clean_df)} rows of data.")

    all_res = []
    for d in clean_df['Date'].unique():
        # Convert numpy datetime back to pandas timestamp for the solver
        ts = pd.to_datetime(d)
        day_data = clean_df[clean_df['Date'] == d]
        
        # Map Hour to FTE
        demand_dict = dict(zip(day_data['Hour'], day_data['FTE_Brut']))
        
        # Solve for the day
        all_res.extend(solve_day(ts, demand_dict))

    if len(all_res) > 0:
        output_df = pd.DataFrame(all_res)
        output_df.to_excel("final_schedule.xlsx", index=False)
        print("\n*** SUCCESS! Check final_schedule.xlsx ***")
    else:
        print("\n*** ERROR: No assignments were generated. Check the text above for warnings. ***")

except Exception as e:
    # This will tell us EXACTLY what went wrong
    import traceback
    print("\n*** CRITICAL ERROR DETAILS ***")
    print(traceback.format_exc())

I found 14 columns in your Excel file.
Column names found: ['Date', 'Day', 'Hour', 'Volume', 'Sizing FTEs', 'Calls/Day/FTE', 'Cases/Day/FTE', 'Service Level %', 'Avg Wait Time (s)', 'Occupancy %', 'P(Wait) %', 'Immediate Answer %', 'Shift', 'Weekend']
Successfully processed 2208 rows of data.
--- Day: 2026-05-01 (Weekday 4) | Agents Available: 50 ---
Status: Optimal
Assignments made: 0
--- Day: 2026-05-02 (Weekday 5) | Agents Available: 15 ---
Status: Optimal
Assignments made: 0
--- Day: 2026-05-03 (Weekday 6) | Agents Available: 15 ---
Status: Optimal
Assignments made: 0
--- Day: 2026-05-04 (Weekday 0) | Agents Available: 50 ---
Status: Optimal
Assignments made: 0
--- Day: 2026-05-05 (Weekday 1) | Agents Available: 50 ---
Status: Optimal
Assignments made: 0
--- Day: 2026-05-06 (Weekday 2) | Agents Available: 50 ---
Status: Optimal
Assignments made: 0
--- Day: 2026-05-07 (Weekday 3) | Agents Available: 50 ---
Status: Optimal
Assignments made: 0
--- Day: 2026-05-08 (Weekday 4) | Agents 

Status: Optimal
Assignments made: 0
--- Day: 2026-07-24 (Weekday 4) | Agents Available: 50 ---
Status: Optimal
Assignments made: 0
--- Day: 2026-07-25 (Weekday 5) | Agents Available: 15 ---
Status: Optimal
Assignments made: 0
--- Day: 2026-07-26 (Weekday 6) | Agents Available: 15 ---
Status: Optimal
Assignments made: 0
--- Day: 2026-07-27 (Weekday 0) | Agents Available: 50 ---
Status: Optimal
Assignments made: 0
--- Day: 2026-07-28 (Weekday 1) | Agents Available: 50 ---
Status: Optimal
Assignments made: 0
--- Day: 2026-07-29 (Weekday 2) | Agents Available: 50 ---
Status: Optimal
Assignments made: 0
--- Day: 2026-07-30 (Weekday 3) | Agents Available: 50 ---
Status: Optimal
Assignments made: 0
--- Day: 2026-07-31 (Weekday 4) | Agents Available: 50 ---
Status: Optimal
Assignments made: 0

*** ERROR: No assignments were generated. Check the text above for warnings. ***


In [14]:
import pulp
import pandas as pd
from datetime import datetime

# --- 1. CONFIGURATION: YOUR AGENT LIST --- # Check these workdays! 0=Mon, 4=Fri, 5=Sat, 6=Sun.
agents_db = {}
for i in range(1, 51):
    if i <= 20: # 1-20: Morning (Mon-Fri)
        agents_db[f"Agent_{i:02d}"] = {"type": "FT", "workdays": [0,1,2,3,4], "window": (0, 9)}
    elif i <= 35: # 21-35: Afternoon (Mon-Fri)
        agents_db[f"Agent_{i:02d}"] = {"type": "FT", "workdays": [0,1,2,3,4], "window": (9, 23)}
    else: # 36-50: Everyone else (Full Week)
        agents_db[f"Agent_{i:02d}"] = {"type": "PT", "workdays": [0,1,2,3,4,5,6], "window": (0, 23)}

def solve_day(date_val, demand_dict):
    weekday = date_val.weekday()
    avail = [a for a, info in agents_db.items() if weekday in info['workdays']]
    
    # DEBUG: See how many agents are available
    print(f"--- Day: {date_val.date()} (Weekday {weekday}) | Agents Available: {len(avail)} ---")
    
    if len(avail) == 0:
        print("!!! WARNING: No agents available for this day of the week.")
        return []

    prob = pulp.LpProblem(f"Sch_{date_val.date()}", pulp.LpMinimize)
    choices = {}
    for a in avail:
        s_min, s_max = agents_db[a]['window']
        for s in range(s_min, s_max + 1):
            choices[(a, s)] = pulp.LpVariable(f"S_{a}_{s}_{date_val.date()}", cat="Binary")

    # Constraint: Max 1 shift
    for a in avail:
        s_min, s_max = agents_db[a]['window']
        prob += pulp.lpSum([choices[(a, s)] for s in range(s_min, s_max + 1)]) <= 1

    # Constraint: Coverage with "Slack" (allows it to finish even if understaffed)
    for h in range(24):
        coverage = []
        for a in avail:
            dur = 8 if agents_db[a]['type'] == "FT" else 4
            s_min, s_max = agents_db[a]['window']
            for s in range(s_min, s_max + 1):
                if s <= h < s + dur:
                    if agents_db[a]['type'] == "FT" and h == (s + 4): # Lunch
                        coverage.append(choices[(a, s)] * 0)
                    else: coverage.append(choices[(a, s)] * 1)
        
        # This 'slack' variable makes the code 'Optimal' even if you don't have enough staff
        slack = pulp.LpVariable(f"Slack_{h}", lowBound=0)
        prob += pulp.lpSum(coverage) + slack >= demand_dict.get(h, 0)

    # Goal: Minimize agents used (you can change this to minimize overstaffing)
    prob += pulp.lpSum([choices[v] for v in choices])
    
    prob.solve(pulp.PULP_CBC_CMD(msg=0))
    
    res = []
    print(f"Status: {pulp.LpStatus[prob.status]}") # Should say 'Optimal'
    
    for (a, s) in choices:
        if pulp.value(choices[(a, s)]) == 1:
            dur = 8 if agents_db[a]['type'] == "FT" else 4
            res.append({"Date": date_val.date(), "Agent": a, "Start": f"{s:02d}:00", "End": f"{(s+dur)%24:02d}:00"})
    
    print(f"Assignments made: {len(res)}")
    return res

# --- UPDATED LOADING AND RUNNING SECTION ---
try:
    # 1. Load the file
    df = pd.read_excel("need.xlsx")
    
    # 2. Let's see what the computer actually sees (Diagnostic)
    actual_col_count = len(df.columns)
    print(f"I found {actual_col_count} columns in your Excel file.")
    print(f"Column names found: {list(df.columns)}")

    # 3. Instead of renaming, we just pick by position:
    # We assume: Col 0 = Date, Col 2 = Hour, Col 3 = FTE_Brut
    # (We skip index 1 because that's usually the 'Day' string)
    
    # Create a cleaned version of the data
    clean_df = pd.DataFrame()
    clean_df['Date'] = pd.to_datetime(df.iloc[:, 0]) # Column A
    clean_df['Hour'] = df.iloc[:, 2]                 # Column C
    clean_df['FTE_Brut'] = df.iloc[:, 4]             # Column E
    
    print(f"Successfully processed {len(clean_df)} rows of data.")

    
    all_res = []
    for d in clean_df['Date'].unique():
        # Convert numpy datetime back to pandas timestamp for the solver
        ts = pd.to_datetime(d)
        day_data = clean_df[clean_df['Date'] == d]
        
        # Map Hour to FTE
        demand_dict = dict(zip(day_data['Hour'], day_data['FTE_Brut']))
        
        # Solve for the day
        all_res.extend(solve_day(ts, demand_dict))

    print (demand_dict)
        
    if len(all_res) > 0:
        output_df = pd.DataFrame(all_res)
        output_df.to_excel("final_schedule.xlsx", index=False)
        print("\n*** SUCCESS! Check final_schedule.xlsx ***")
    else:
        print("\n*** ERROR: No assignments were generated. Check the text above for warnings. ***")

except Exception as e:
    # This will tell us EXACTLY what went wrong
    import traceback
    print("\n*** CRITICAL ERROR DETAILS ***")
    print(traceback.format_exc())

I found 14 columns in your Excel file.
Column names found: ['Date', 'Day', 'Hour', 'Volume', 'Sizing FTEs', 'Calls/Day/FTE', 'Cases/Day/FTE', 'Service Level %', 'Avg Wait Time (s)', 'Occupancy %', 'P(Wait) %', 'Immediate Answer %', 'Shift', 'Weekend']
Successfully processed 2208 rows of data.
--- Day: 2026-05-01 (Weekday 4) | Agents Available: 50 ---
Status: Optimal
Assignments made: 0
--- Day: 2026-05-02 (Weekday 5) | Agents Available: 15 ---
Status: Optimal
Assignments made: 0
--- Day: 2026-05-03 (Weekday 6) | Agents Available: 15 ---
Status: Optimal
Assignments made: 0
--- Day: 2026-05-04 (Weekday 0) | Agents Available: 50 ---
Status: Optimal
Assignments made: 0
--- Day: 2026-05-05 (Weekday 1) | Agents Available: 50 ---
Status: Optimal
Assignments made: 0
--- Day: 2026-05-06 (Weekday 2) | Agents Available: 50 ---
Status: Optimal
Assignments made: 0
--- Day: 2026-05-07 (Weekday 3) | Agents Available: 50 ---
Status: Optimal
Assignments made: 0
--- Day: 2026-05-08 (Weekday 4) | Agents 

Status: Optimal
Assignments made: 0
--- Day: 2026-07-24 (Weekday 4) | Agents Available: 50 ---
Status: Optimal
Assignments made: 0
--- Day: 2026-07-25 (Weekday 5) | Agents Available: 15 ---
Status: Optimal
Assignments made: 0
--- Day: 2026-07-26 (Weekday 6) | Agents Available: 15 ---
Status: Optimal
Assignments made: 0
--- Day: 2026-07-27 (Weekday 0) | Agents Available: 50 ---
Status: Optimal
Assignments made: 0
--- Day: 2026-07-28 (Weekday 1) | Agents Available: 50 ---
Status: Optimal
Assignments made: 0
--- Day: 2026-07-29 (Weekday 2) | Agents Available: 50 ---
Status: Optimal
Assignments made: 0
--- Day: 2026-07-30 (Weekday 3) | Agents Available: 50 ---
Status: Optimal
Assignments made: 0
--- Day: 2026-07-31 (Weekday 4) | Agents Available: 50 ---
Status: Optimal
Assignments made: 0
{'00:00': 8.3, '01:00': 5.7, '02:00': 5.7, '03:00': 5.7, '04:00': 5.7, '05:00': 5.7, '06:00': 8.3, '07:00': 11.0, '08:00': 12.3, '09:00': 16.3, '10:00': 19.0, '11:00': 20.3, '12:00': 20.3, '13:00': 20.3, 

In [15]:
import pulp
import pandas as pd
from datetime import datetime

# --- 1. CONFIGURATION: YOUR AGENT LIST --- # Check these workdays! 0=Mon, 4=Fri, 5=Sat, 6=Sun.
agents_db = {}
for i in range(1, 51):
    if i <= 20: # 1-20: Morning (Mon-Fri)
        agents_db[f"Agent_{i:02d}"] = {"type": "FT", "workdays": [0,1,2,3,4], "window": (0, 9)}
    elif i <= 35: # 21-35: Afternoon (Mon-Fri)
        agents_db[f"Agent_{i:02d}"] = {"type": "FT", "workdays": [0,1,2,3,4], "window": (9, 23)}
    else: # 36-50: Everyone else (Full Week)
        agents_db[f"Agent_{i:02d}"] = {"type": "PT", "workdays": [0,1,2,3,4,5,6], "window": (0, 23)}

def solve_day(date_val, demand_dict):
    weekday = date_val.weekday()
    avail = [a for a, info in agents_db.items() if weekday in info['workdays']]
    
    # DEBUG: See how many agents are available
    print(f"--- Day: {date_val.date()} (Weekday {weekday}) | Agents Available: {len(avail)} ---")
    
    if len(avail) == 0:
        print("!!! WARNING: No agents available for this day of the week.")
        return []

    prob = pulp.LpProblem(f"Sch_{date_val.date()}", pulp.LpMinimize)
    choices = {}
    for a in avail:
        s_min, s_max = agents_db[a]['window']
        for s in range(s_min, s_max + 1):
            choices[(a, s)] = pulp.LpVariable(f"S_{a}_{s}_{date_val.date()}", cat="Binary")

    # Constraint: Max 1 shift
    for a in avail:
        s_min, s_max = agents_db[a]['window']
        prob += pulp.lpSum([choices[(a, s)] for s in range(s_min, s_max + 1)]) <= 1

    # Constraint: Coverage with "Slack" (allows it to finish even if understaffed)
    for h in range(24):
        coverage = []
        for a in avail:
            dur = 8 if agents_db[a]['type'] == "FT" else 4
            s_min, s_max = agents_db[a]['window']
            for s in range(s_min, s_max + 1):
                if s <= h < s + dur:
                    if agents_db[a]['type'] == "FT" and h == (s + 4): # Lunch
                        coverage.append(choices[(a, s)] * 0)
                    else: coverage.append(choices[(a, s)] * 1)
        
        # This 'slack' variable makes the code 'Optimal' even if you don't have enough staff
        slack = pulp.LpVariable(f"Slack_{h}", lowBound=0)
        prob += pulp.lpSum(coverage) + slack >= demand_dict.get(h, 0)

    # Goal: Minimize agents used (you can change this to minimize overstaffing)
    prob += pulp.lpSum([choices[v] for v in choices])
    
    prob.solve(pulp.PULP_CBC_CMD(msg=0))
    
    res = []
    print(f"Status: {pulp.LpStatus[prob.status]}") # Should say 'Optimal'
    
    for (a, s) in choices:
        if pulp.value(choices[(a, s)]) == 1:
            dur = 8 if agents_db[a]['type'] == "FT" else 4
            res.append({"Date": date_val.date(), "Agent": a, "Start": f"{s:02d}:00", "End": f"{(s+dur)%24:02d}:00"})
    
    print(f"Assignments made: {len(res)}")
    return res

# --- UPDATED LOADING AND RUNNING SECTION ---
try:
    # 1. Load the file
    df = pd.read_excel("need.xlsx")
    
    # 2. Let's see what the computer actually sees (Diagnostic)
    actual_col_count = len(df.columns)
    print(f"I found {actual_col_count} columns in your Excel file.")
    print(f"Column names found: {list(df.columns)}")

    # 3. Instead of renaming, we just pick by position:
    # We assume: Col 0 = Date, Col 2 = Hour, Col 3 = FTE_Brut
    # (We skip index 1 because that's usually the 'Day' string)
    
    # Create a cleaned version of the data
    clean_df = pd.DataFrame()
    clean_df['Date'] = pd.to_datetime(df.iloc[:, 0]) # Column A
    clean_df['Hour'] = df.iloc[:, 2]                 # Column C
    clean_df['FTE_Brut'] = df.iloc[:, 4]             # Column E
    
    print(f"Successfully processed {len(clean_df)} rows of data.")

    
    all_res = []
    for d in clean_df['Date'].unique():
        # Convert numpy datetime back to pandas timestamp for the solver
        ts = pd.to_datetime(d)
        day_data = clean_df[clean_df['Date'] == d]
        
        # Map Hour to FTE
        demand_dict = dict(zip(day_data['Hour'], day_data['FTE_Brut']))
        
        
        #---DEBUG PRINT----
        print(f"Total Demand for {ts.date()}:{sum(demand_dict.values())}")
        print(f"Hours with demand:{list(demand_dict.keys())}")
    
        # Solve for the day
        all_res.extend(solve_day(ts, demand_dict))

    print (demand_dict)
        
    if len(all_res) > 0:
        output_df = pd.DataFrame(all_res)
        output_df.to_excel("final_schedule.xlsx", index=False)
        print("\n*** SUCCESS! Check final_schedule.xlsx ***")
    else:
        print("\n*** ERROR: No assignments were generated. Check the text above for warnings. ***")

except Exception as e:
    # This will tell us EXACTLY what went wrong
    import traceback
    print("\n*** CRITICAL ERROR DETAILS ***")
    print(traceback.format_exc())

I found 14 columns in your Excel file.
Column names found: ['Date', 'Day', 'Hour', 'Volume', 'Sizing FTEs', 'Calls/Day/FTE', 'Cases/Day/FTE', 'Service Level %', 'Avg Wait Time (s)', 'Occupancy %', 'P(Wait) %', 'Immediate Answer %', 'Shift', 'Weekend']
Successfully processed 2208 rows of data.
Total Demand for 2026-05-01:266.7
Hours with demand:['00:00', '01:00', '02:00', '03:00', '04:00', '05:00', '06:00', '07:00', '08:00', '09:00', '10:00', '11:00', '12:00', '13:00', '14:00', '15:00', '16:00', '17:00', '18:00', '19:00', '20:00', '21:00', '22:00', '23:00']
--- Day: 2026-05-01 (Weekday 4) | Agents Available: 50 ---
Status: Optimal
Assignments made: 0
Total Demand for 2026-05-02:198.60000000000002
Hours with demand:['00:00', '01:00', '02:00', '03:00', '04:00', '05:00', '06:00', '07:00', '08:00', '09:00', '10:00', '11:00', '12:00', '13:00', '14:00', '15:00', '16:00', '17:00', '18:00', '19:00', '20:00', '21:00', '22:00', '23:00']
--- Day: 2026-05-02 (Weekday 5) | Agents Available: 15 ---
S

Status: Optimal
Assignments made: 0
Total Demand for 2026-05-23:198.60000000000002
Hours with demand:['00:00', '01:00', '02:00', '03:00', '04:00', '05:00', '06:00', '07:00', '08:00', '09:00', '10:00', '11:00', '12:00', '13:00', '14:00', '15:00', '16:00', '17:00', '18:00', '19:00', '20:00', '21:00', '22:00', '23:00']
--- Day: 2026-05-23 (Weekday 5) | Agents Available: 15 ---
Status: Optimal
Assignments made: 0
Total Demand for 2026-05-24:201.20000000000002
Hours with demand:['00:00', '01:00', '02:00', '03:00', '04:00', '05:00', '06:00', '07:00', '08:00', '09:00', '10:00', '11:00', '12:00', '13:00', '14:00', '15:00', '16:00', '17:00', '18:00', '19:00', '20:00', '21:00', '22:00', '23:00']
--- Day: 2026-05-24 (Weekday 6) | Agents Available: 15 ---
Status: Optimal
Assignments made: 0
Total Demand for 2026-05-25:273.3999999999999
Hours with demand:['00:00', '01:00', '02:00', '03:00', '04:00', '05:00', '06:00', '07:00', '08:00', '09:00', '10:00', '11:00', '12:00', '13:00', '14:00', '15:00', '

Status: Optimal
Assignments made: 0
Total Demand for 2026-06-15:242.5
Hours with demand:['00:00', '01:00', '02:00', '03:00', '04:00', '05:00', '06:00', '07:00', '08:00', '09:00', '10:00', '11:00', '12:00', '13:00', '14:00', '15:00', '16:00', '17:00', '18:00', '19:00', '20:00', '21:00', '22:00', '23:00']
--- Day: 2026-06-15 (Weekday 0) | Agents Available: 50 ---
Status: Optimal
Assignments made: 0
Total Demand for 2026-06-16:237.1
Hours with demand:['00:00', '01:00', '02:00', '03:00', '04:00', '05:00', '06:00', '07:00', '08:00', '09:00', '10:00', '11:00', '12:00', '13:00', '14:00', '15:00', '16:00', '17:00', '18:00', '19:00', '20:00', '21:00', '22:00', '23:00']
--- Day: 2026-06-16 (Weekday 1) | Agents Available: 50 ---
Status: Optimal
Assignments made: 0
Total Demand for 2026-06-17:256.00000000000006
Hours with demand:['00:00', '01:00', '02:00', '03:00', '04:00', '05:00', '06:00', '07:00', '08:00', '09:00', '10:00', '11:00', '12:00', '13:00', '14:00', '15:00', '16:00', '17:00', '18:00',

Status: Optimal
Assignments made: 0
Total Demand for 2026-07-08:309.5
Hours with demand:['00:00', '01:00', '02:00', '03:00', '04:00', '05:00', '06:00', '07:00', '08:00', '09:00', '10:00', '11:00', '12:00', '13:00', '14:00', '15:00', '16:00', '17:00', '18:00', '19:00', '20:00', '21:00', '22:00', '23:00']
--- Day: 2026-07-08 (Weekday 2) | Agents Available: 50 ---
Status: Optimal
Assignments made: 0
Total Demand for 2026-07-09:304.09999999999997
Hours with demand:['00:00', '01:00', '02:00', '03:00', '04:00', '05:00', '06:00', '07:00', '08:00', '09:00', '10:00', '11:00', '12:00', '13:00', '14:00', '15:00', '16:00', '17:00', '18:00', '19:00', '20:00', '21:00', '22:00', '23:00']
--- Day: 2026-07-09 (Weekday 3) | Agents Available: 50 ---
Status: Optimal
Assignments made: 0
Total Demand for 2026-07-10:284.0
Hours with demand:['00:00', '01:00', '02:00', '03:00', '04:00', '05:00', '06:00', '07:00', '08:00', '09:00', '10:00', '11:00', '12:00', '13:00', '14:00', '15:00', '16:00', '17:00', '18:00',

Status: Optimal
Assignments made: 0
Total Demand for 2026-07-31:284.0
Hours with demand:['00:00', '01:00', '02:00', '03:00', '04:00', '05:00', '06:00', '07:00', '08:00', '09:00', '10:00', '11:00', '12:00', '13:00', '14:00', '15:00', '16:00', '17:00', '18:00', '19:00', '20:00', '21:00', '22:00', '23:00']
--- Day: 2026-07-31 (Weekday 4) | Agents Available: 50 ---
Status: Optimal
Assignments made: 0
{'00:00': 8.3, '01:00': 5.7, '02:00': 5.7, '03:00': 5.7, '04:00': 5.7, '05:00': 5.7, '06:00': 8.3, '07:00': 11.0, '08:00': 12.3, '09:00': 16.3, '10:00': 19.0, '11:00': 20.3, '12:00': 20.3, '13:00': 20.3, '14:00': 19.0, '15:00': 17.7, '16:00': 16.3, '17:00': 13.7, '18:00': 12.3, '19:00': 11.0, '20:00': 9.7, '21:00': 7.0, '22:00': 7.0, '23:00': 5.7}

*** ERROR: No assignments were generated. Check the text above for warnings. ***


In [16]:
import pulp
import pandas as pd
from datetime import datetime

# --- 1. CONFIGURATION: YOUR AGENT LIST --- # Check these workdays! 0=Mon, 4=Fri, 5=Sat, 6=Sun.
agents_db = {}
for i in range(1, 51):
    if i <= 20: # 1-20: Morning (Mon-Fri)
        agents_db[f"Agent_{i:02d}"] = {"type": "FT", "workdays": [0,1,2,3,4], "window": (0, 9)}
    elif i <= 35: # 21-35: Afternoon (Mon-Fri)
        agents_db[f"Agent_{i:02d}"] = {"type": "FT", "workdays": [0,1,2,3,4], "window": (9, 23)}
    else: # 36-50: Everyone else (Full Week)
        agents_db[f"Agent_{i:02d}"] = {"type": "PT", "workdays": [0,1,2,3,4,5,6], "window": (0, 23)}

def solve_day(date_val, demand_dict):
    weekday = date_val.weekday()
    avail = [a for a, info in agents_db.items() if weekday in info['workdays']]
    
    # DEBUG: See how many agents are available
    print(f"--- Day: {date_val.date()} (Weekday {weekday}) | Agents Available: {len(avail)} ---")
    
    if len(avail) == 0:
        print("!!! WARNING: No agents available for this day of the week.")
        return []

    prob = pulp.LpProblem(f"Sch_{date_val.date()}", pulp.LpMinimize)
    choices = {}
    for a in avail:
        s_min, s_max = agents_db[a]['window']
        for s in range(s_min, s_max + 1):
            choices[(a, s)] = pulp.LpVariable(f"S_{a}_{s}_{date_val.date()}", cat="Binary")

    # Constraint: Max 1 shift
    for a in avail:
        s_min, s_max = agents_db[a]['window']
        prob += pulp.lpSum([choices[(a, s)] for s in range(s_min, s_max + 1)]) <= 1

    # Constraint: Coverage with "Slack" (allows it to finish even if understaffed)
    for h in range(24):
        coverage = []
        for a in avail:
            dur = 8 if agents_db[a]['type'] == "FT" else 4
            s_min, s_max = agents_db[a]['window']
            for s in range(s_min, s_max + 1):
                if s <= h < s + dur:
                    if agents_db[a]['type'] == "FT" and h == (s + 4): # Lunch
                        coverage.append(choices[(a, s)] * 0)
                    else: coverage.append(choices[(a, s)] * 1)
        
        # This 'slack' variable makes the code 'Optimal' even if you don't have enough staff
        slack = pulp.LpVariable(f"Slack_{h}", lowBound=0)
        prob += pulp.lpSum(coverage) + slack >= demand_dict.get(h, 0)

    # Goal: Minimize agents used (you can change this to minimize overstaffing)
    prob += pulp.lpSum([choices[v] for v in choices])
    
    prob.solve(pulp.PULP_CBC_CMD(msg=0))
    
    res = []
    print(f"Status: {pulp.LpStatus[prob.status]}") # Should say 'Optimal'
    
    for (a, s) in choices:
        if pulp.value(choices[(a, s)]) == 1:
            dur = 8 if agents_db[a]['type'] == "FT" else 4
            res.append({"Date": date_val.date(), "Agent": a, "Start": f"{s:02d}:00", "End": f"{(s+dur)%24:02d}:00"})
    
    print(f"Assignments made: {len(res)}")
    return res

# --- UPDATED LOADING AND RUNNING SECTION ---
try:
    # 1. Load the file
    df = pd.read_excel("need.xlsx")
    
    # 2. Let's see what the computer actually sees (Diagnostic)
    actual_col_count = len(df.columns)
    print(f"I found {actual_col_count} columns in your Excel file.")
    print(f"Column names found: {list(df.columns)}")

    # 3. Instead of renaming, we just pick by position:
    # We assume: Col 0 = Date, Col 2 = Hour, Col 3 = FTE_Brut
    # (We skip index 1 because that's usually the 'Day' string)
    
    # Create a cleaned version of the data
    clean_df = pd.DataFrame()
    clean_df['Date'] = pd.to_datetime(df.iloc[:, 0]) # Column A
    clean_df['Hour'] = df.iloc[:, 2]                 # Column C
    clean_df['FTE_Brut'] = df.iloc[:, 4]             # Column E
    
    print(f"Successfully processed {len(clean_df)} rows of data.")

    
    all_res = []
    for d in clean_df['Date'].unique():
        # Convert numpy datetime back to pandas timestamp for the solver
        ts = pd.to_datetime(d)
        day_data = clean_df[clean_df['Date'] == d]
        
        # Map Hour to FTE
        demand_dict = dict(zip(day_data['Hour'], day_data['FTE_Brut']))
        
        
        #---DEBUG PRINT----
        print(f"Total Demand for {ts.date()}:{sum(demand_dict.values())}")
        print(f"Hours with demand:{list(demand_dict.keys())}")
    
        # Solve for the day
        all_res.extend(solve_day(ts, demand_dict))

    print (demand_dict)
        
    if len(all_res) > 0:
        output_df = pd.DataFrame(all_res)
        output_df.to_excel("final_schedule.xlsx", index=False)
        print("\n*** SUCCESS! Check final_schedule.xlsx ***")
    else:
        print("\n*** ERROR: No assignments were generated. Check the text above for warnings. ***")

except Exception as e:
    # This will tell us EXACTLY what went wrong
    import traceback
    print("\n*** CRITICAL ERROR DETAILS ***")
    print(traceback.format_exc())

I found 14 columns in your Excel file.
Column names found: ['Date', 'Day', 'Hour', 'Volume', 'Sizing FTEs', 'Calls/Day/FTE', 'Cases/Day/FTE', 'Service Level %', 'Avg Wait Time (s)', 'Occupancy %', 'P(Wait) %', 'Immediate Answer %', 'Shift', 'Weekend']
Successfully processed 2208 rows of data.
Total Demand for 2026-05-01:41.03076923076924
Hours with demand:['00:00', '01:00', '02:00', '03:00', '04:00', '05:00', '06:00', '07:00', '08:00', '09:00', '10:00', '11:00', '12:00', '13:00', '14:00', '15:00', '16:00', '17:00', '18:00', '19:00', '20:00', '21:00', '22:00', '23:00']
--- Day: 2026-05-01 (Weekday 4) | Agents Available: 50 ---
Status: Optimal
Assignments made: 0
Total Demand for 2026-05-02:30.553846153846155
Hours with demand:['00:00', '01:00', '02:00', '03:00', '04:00', '05:00', '06:00', '07:00', '08:00', '09:00', '10:00', '11:00', '12:00', '13:00', '14:00', '15:00', '16:00', '17:00', '18:00', '19:00', '20:00', '21:00', '22:00', '23:00']
--- Day: 2026-05-02 (Weekday 5) | Agents Availab

Status: Optimal
Assignments made: 0
Total Demand for 2026-05-23:30.553846153846155
Hours with demand:['00:00', '01:00', '02:00', '03:00', '04:00', '05:00', '06:00', '07:00', '08:00', '09:00', '10:00', '11:00', '12:00', '13:00', '14:00', '15:00', '16:00', '17:00', '18:00', '19:00', '20:00', '21:00', '22:00', '23:00']
--- Day: 2026-05-23 (Weekday 5) | Agents Available: 15 ---
Status: Optimal
Assignments made: 0
Total Demand for 2026-05-24:30.953846153846154
Hours with demand:['00:00', '01:00', '02:00', '03:00', '04:00', '05:00', '06:00', '07:00', '08:00', '09:00', '10:00', '11:00', '12:00', '13:00', '14:00', '15:00', '16:00', '17:00', '18:00', '19:00', '20:00', '21:00', '22:00', '23:00']
--- Day: 2026-05-24 (Weekday 6) | Agents Available: 15 ---
Status: Optimal
Assignments made: 0
Total Demand for 2026-05-25:42.06153846153847
Hours with demand:['00:00', '01:00', '02:00', '03:00', '04:00', '05:00', '06:00', '07:00', '08:00', '09:00', '10:00', '11:00', '12:00', '13:00', '14:00', '15:00', '

Status: Optimal
Assignments made: 0
Total Demand for 2026-06-14:28.061538461538465
Hours with demand:['00:00', '01:00', '02:00', '03:00', '04:00', '05:00', '06:00', '07:00', '08:00', '09:00', '10:00', '11:00', '12:00', '13:00', '14:00', '15:00', '16:00', '17:00', '18:00', '19:00', '20:00', '21:00', '22:00', '23:00']
--- Day: 2026-06-14 (Weekday 6) | Agents Available: 15 ---
Status: Optimal
Assignments made: 0
Total Demand for 2026-06-15:37.307692307692314
Hours with demand:['00:00', '01:00', '02:00', '03:00', '04:00', '05:00', '06:00', '07:00', '08:00', '09:00', '10:00', '11:00', '12:00', '13:00', '14:00', '15:00', '16:00', '17:00', '18:00', '19:00', '20:00', '21:00', '22:00', '23:00']
--- Day: 2026-06-15 (Weekday 0) | Agents Available: 50 ---
Status: Optimal
Assignments made: 0
Total Demand for 2026-06-16:36.47692307692308
Hours with demand:['00:00', '01:00', '02:00', '03:00', '04:00', '05:00', '06:00', '07:00', '08:00', '09:00', '10:00', '11:00', '12:00', '13:00', '14:00', '15:00', '

Status: Optimal
Assignments made: 0
Total Demand for 2026-07-06:44.9076923076923
Hours with demand:['00:00', '01:00', '02:00', '03:00', '04:00', '05:00', '06:00', '07:00', '08:00', '09:00', '10:00', '11:00', '12:00', '13:00', '14:00', '15:00', '16:00', '17:00', '18:00', '19:00', '20:00', '21:00', '22:00', '23:00']
--- Day: 2026-07-06 (Weekday 0) | Agents Available: 50 ---
Status: Optimal
Assignments made: 0
Total Demand for 2026-07-07:43.6923076923077
Hours with demand:['00:00', '01:00', '02:00', '03:00', '04:00', '05:00', '06:00', '07:00', '08:00', '09:00', '10:00', '11:00', '12:00', '13:00', '14:00', '15:00', '16:00', '17:00', '18:00', '19:00', '20:00', '21:00', '22:00', '23:00']
--- Day: 2026-07-07 (Weekday 1) | Agents Available: 50 ---
Status: Optimal
Assignments made: 0
Total Demand for 2026-07-08:47.615384615384606
Hours with demand:['00:00', '01:00', '02:00', '03:00', '04:00', '05:00', '06:00', '07:00', '08:00', '09:00', '10:00', '11:00', '12:00', '13:00', '14:00', '15:00', '16:

Status: Optimal
Assignments made: 0
Total Demand for 2026-07-28:43.6923076923077
Hours with demand:['00:00', '01:00', '02:00', '03:00', '04:00', '05:00', '06:00', '07:00', '08:00', '09:00', '10:00', '11:00', '12:00', '13:00', '14:00', '15:00', '16:00', '17:00', '18:00', '19:00', '20:00', '21:00', '22:00', '23:00']
--- Day: 2026-07-28 (Weekday 1) | Agents Available: 50 ---
Status: Optimal
Assignments made: 0
Total Demand for 2026-07-29:47.615384615384606
Hours with demand:['00:00', '01:00', '02:00', '03:00', '04:00', '05:00', '06:00', '07:00', '08:00', '09:00', '10:00', '11:00', '12:00', '13:00', '14:00', '15:00', '16:00', '17:00', '18:00', '19:00', '20:00', '21:00', '22:00', '23:00']
--- Day: 2026-07-29 (Weekday 2) | Agents Available: 50 ---
Status: Optimal
Assignments made: 0
Total Demand for 2026-07-30:46.78461538461537
Hours with demand:['00:00', '01:00', '02:00', '03:00', '04:00', '05:00', '06:00', '07:00', '08:00', '09:00', '10:00', '11:00', '12:00', '13:00', '14:00', '15:00', '16

In [18]:
import pulp
import pandas as pd
from datetime import datetime

# --- 1. CONFIGURATION: YOUR AGENT LIST --- # Check these workdays! 0=Mon, 4=Fri, 5=Sat, 6=Sun.
agents_db = {}
for i in range(1, 51):
    if i <= 20: # 1-20: Morning (Mon-Fri)
        agents_db[f"Agent_{i:02d}"] = {"type": "FT", "workdays": [0,1,2,3,4], "window": (0, 9)}
    elif i <= 35: # 21-35: Afternoon (Mon-Fri)
        agents_db[f"Agent_{i:02d}"] = {"type": "FT", "workdays": [0,1,2,3,4], "window": (9, 23)}
    else: # 36-50: Everyone else (Full Week)
        agents_db[f"Agent_{i:02d}"] = {"type": "PT", "workdays": [0,1,2,3,4,5,6], "window": (0, 23)}

def solve_day(date_val, demand_dict):
    weekday = date_val.weekday()
    avail = [a for a, info in agents_db.items() if weekday in info['workdays']]
    
    # DEBUG: See how many agents are available
    print(f"--- Day: {date_val.date()} (Weekday {weekday}) | Agents Available: {len(avail)} ---")
    
    if len(avail) == 0:
        print("!!! WARNING: No agents available for this day of the week.")
        return []

    prob = pulp.LpProblem(f"Sch_{date_val.date()}", pulp.LpMinimize)
    choices = {}
    for a in avail:
        s_min, s_max = agents_db[a]['window']
        for s in range(s_min, s_max + 1):
            choices[(a, s)] = pulp.LpVariable(f"S_{a}_{s}_{date_val.date()}", cat="Binary")

    # Constraint: Max 1 shift
    for a in avail:
        s_min, s_max = agents_db[a]['window']
        prob += pulp.lpSum([choices[(a, s)] for s in range(s_min, s_max + 1)]) <= 1

    # Constraint: Coverage with "Slack" (allows it to finish even if understaffed)
    for h in range(24):
        coverage = []
        for a in avail:
            dur = 8 if agents_db[a]['type'] == "FT" else 4
            s_min, s_max = agents_db[a]['window']
            for s in range(s_min, s_max + 1):
                if s <= h < s + dur:
                    if agents_db[a]['type'] == "FT" and h == (s + 4): # Lunch
                        coverage.append(choices[(a, s)] * 0)
                    else: coverage.append(choices[(a, s)] * 1)
        
        # This 'slack' variable makes the code 'Optimal' even if you don't have enough staff
        slack = pulp.LpVariable(f"Slack_{h}", lowBound=0)
        prob += pulp.lpSum(coverage) + slack >= demand_dict.get(h, 0)

    # Goal: Minimize agents used (you can change this to minimize overstaffing)
    prob += pulp.lpSum([choices[v] for v in choices])
    
    prob.solve(pulp.PULP_CBC_CMD(msg=0))
    
    res = []
    print(f"Status: {pulp.LpStatus[prob.status]}") # Should say 'Optimal'
    
    for (a, s) in choices:
        if pulp.value(choices[(a, s)]) == 1:
            dur = 8 if agents_db[a]['type'] == "FT" else 4
            res.append({"Date": date_val.date(), "Agent": a, "Start": f"{s:02d}:00", "End": f"{(s+dur)%24:02d}:00"})
    
    print(f"Assignments made: {len(res)}")
    return res

# --- UPDATED LOADING AND RUNNING SECTION ---
try:
    # 1. Load the file
    df = pd.read_excel("need.xlsx")
    
    # 2. Let's see what the computer actually sees (Diagnostic)
    actual_col_count = len(df.columns)
    print(f"I found {actual_col_count} columns in your Excel file.")
    print(f"Column names found: {list(df.columns)}")

       # 3. Create a cleaned version of the data
    clean_df = pd.DataFrame()
   
   # Column A: Date
    clean_df['Date'] = pd.to_datetime(df.iloc[:, 0]) 
   
   # Column C: Hour (Converting "09:00" string to integer 9)
   # We convert to string, split at the colon, take the first part, and make it a number
    clean_df['Hour'] = pd.to_numeric(df.iloc[:, 2].astype(str).str.split(':').str[0], errors='coerce')
   
   # Column D: FTE_Brut (Ensuring it is a number, not text)
    clean_df['FTE_Brut'] = pd.to_numeric(df.iloc[:, 4], errors='coerce').fillna(0)

   # Clean up any rows that aren't valid numbers
    clean_df = clean_df.dropna(subset=['Hour'])


    
    print(f"Successfully processed {len(clean_df)} rows of data.")

    
    all_res = []
    for d in clean_df['Date'].unique():
        # Convert numpy datetime back to pandas timestamp for the solver
        ts = pd.to_datetime(d)
        day_data = clean_df[clean_df['Date'] == d]
        
        # Map Hour to FTE
        demand_dict = dict(zip(day_data['Hour'], day_data['FTE_Brut']))
        
        
        #---DEBUG PRINT----
        print(f"Total Demand for {ts.date()}:{sum(demand_dict.values())}")
        print(f"Hours with demand:{list(demand_dict.keys())}")
    
        # Solve for the day
        all_res.extend(solve_day(ts, demand_dict))

    print (demand_dict)
        
    if len(all_res) > 0:
        output_df = pd.DataFrame(all_res)
        output_df.to_excel("final_schedule.xlsx", index=False)
        print("\n*** SUCCESS! Check final_schedule.xlsx ***")
    else:
        print("\n*** ERROR: No assignments were generated. Check the text above for warnings. ***")

except Exception as e:
    # This will tell us EXACTLY what went wrong
    import traceback
    print("\n*** CRITICAL ERROR DETAILS ***")
    print(traceback.format_exc())

I found 15 columns in your Excel file.
Column names found: ['Date', 'Day', 'Hour', 'Volume', 'Sizing FTEs', 'Calls/Day/FTE', 'Cases/Day/FTE', 'Service Level %', 'Avg Wait Time (s)', 'Occupancy %', 'P(Wait) %', 'Immediate Answer %', 'Shift', 'Weekend', 'Unnamed: 14']
Successfully processed 2208 rows of data.
Total Demand for 2026-05-01:82.06153846153848
Hours with demand:[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23]
--- Day: 2026-05-01 (Weekday 4) | Agents Available: 50 ---
Status: Optimal
Assignments made: 0
Total Demand for 2026-05-02:61.10769230769231
Hours with demand:[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23]
--- Day: 2026-05-02 (Weekday 5) | Agents Available: 15 ---
Status: Optimal
Assignments made: 0
Total Demand for 2026-05-03:61.90769230769231
Hours with demand:[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23]
--- Day: 2026-05-03 (Weekday 6) | Agents Availab

Status: Optimal
Assignments made: 0
Total Demand for 2026-06-03:78.76923076923079
Hours with demand:[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23]
--- Day: 2026-06-03 (Weekday 2) | Agents Available: 50 ---
Status: Optimal
Assignments made: 0
Total Demand for 2026-06-04:77.13846153846156
Hours with demand:[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23]
--- Day: 2026-06-04 (Weekday 3) | Agents Available: 50 ---
Status: Optimal
Assignments made: 0
Total Demand for 2026-06-05:73.35384615384616
Hours with demand:[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23]
--- Day: 2026-06-05 (Weekday 4) | Agents Available: 50 ---
Status: Optimal
Assignments made: 0
Total Demand for 2026-06-06:56.12307692307693
Hours with demand:[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23]
--- Day: 2026-06-06 (Weekday 5) | Agents Available: 15 ---
Status: Optimal


Status: Optimal
Assignments made: 0
Total Demand for 2026-07-07:87.3846153846154
Hours with demand:[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23]
--- Day: 2026-07-07 (Weekday 1) | Agents Available: 50 ---
Status: Optimal
Assignments made: 0
Total Demand for 2026-07-08:95.23076923076921
Hours with demand:[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23]
--- Day: 2026-07-08 (Weekday 2) | Agents Available: 50 ---
Status: Optimal
Assignments made: 0
Total Demand for 2026-07-09:93.56923076923074
Hours with demand:[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23]
--- Day: 2026-07-09 (Weekday 3) | Agents Available: 50 ---
Status: Optimal
Assignments made: 0
Total Demand for 2026-07-10:87.3846153846154
Hours with demand:[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23]
--- Day: 2026-07-10 (Weekday 4) | Agents Available: 50 ---
Status: Optimal
As

In [ ]:
import pulp
import pandas as pd
from datetime import datetime

# --- 1. CONFIGURATION: YOUR AGENT LIST --- # Check these workdays! 0=Mon, 4=Fri, 5=Sat, 6=Sun.
agents_db = {}
for i in range(1, 51):
    if i <= 20: # 1-20: Morning (Mon-Fri)
        agents_db[f"Agent_{i:02d}"] = {"type": "FT", "workdays": [0,1,2,3,4], "window": (0, 9)}
    elif i <= 35: # 21-35: Afternoon (Mon-Fri)
        agents_db[f"Agent_{i:02d}"] = {"type": "FT", "workdays": [0,1,2,3,4], "window": (9, 23)}
    else: # 36-50: Everyone else (Full Week)
        agents_db[f"Agent_{i:02d}"] = {"type": "PT", "workdays": [0,1,2,3,4,5,6], "window": (0, 23)}

def solve_day(date_val, demand_dict):
    weekday = date_val.weekday()
    avail = [a for a, info in agents_db.items() if weekday in info['workdays']]
    
    # DEBUG: See how many agents are available
    print(f"--- Day: {date_val.date()} (Weekday {weekday}) | Agents Available: {len(avail)} ---")
    
    if len(avail) == 0:
        print("!!! WARNING: No agents available for this day of the week.")
        return []

    prob = pulp.LpProblem(f"Sch_{date_val.date()}", pulp.LpMinimize)
    slack_vars = {}
    choices = {}
    for a in avail:
        s_min, s_max = agents_db[a]['window']
        for s in range(s_min, s_max + 1):
            choices[(a, s)] = pulp.LpVariable(f"S_{a}_{s}_{date_val.date()}", cat="Binary")

    # Constraint: Max 1 shift
    for a in avail:
        s_min, s_max = agents_db[a]['window']
        prob += pulp.lpSum([choices[(a, s)] for s in range(s_min, s_max + 1)]) <= 1

    # Constraint: Coverage with "Slack" (allows it to finish even if understaffed)
    for h in range(24):
        coverage = []
        for a in avail:
            dur = 8 if agents_db[a]['type'] == "FT" else 4
            s_min, s_max = agents_db[a]['window']
            for s in range(s_min, s_max + 1):
                if s <= h < s + dur:
                    if agents_db[a]['type'] == "FT" and h == (s + 4): # Lunch
                        coverage.append(choices[(a, s)] * 0)
                    else: coverage.append(choices[(a, s)] * 1)
        
        # This 'slack' variable makes the code 'Optimal' even if you don't have enough staff
        slack = pulp.LpVariable(f"Slack_{h}", lowBound=0)
        slack_vars[h]=slack
        prob += pulp.lpSum(coverage) + slack >= demand_dict.get(h, 0)

   # Goal: Minimize agents used AND heavily penalize understaffing (Slack)
   # We multiply slack by 100 to make it "expensive" for the computer to ignore demand
    prob += pulp.lpSum([choices[v] for v in choices]) + pulp.lpSum([slack_vars[h] * 100 for h in range(24)])
    prob.solve(pulp.PULP_CBC_CMD(msg=0))
    
    res = []
    print(f"Status: {pulp.LpStatus[prob.status]}") # Should say 'Optimal'
    
    for (a, s) in choices:
        if pulp.value(choices[(a, s)]) == 1:
            dur = 8 if agents_db[a]['type'] == "FT" else 4
            res.append({"Date": date_val.date(), "Agent": a, "Start": f"{s:02d}:00", "End": f"{(s+dur)%24:02d}:00"})
    
    print(f"Assignments made: {len(res)}")
    return res

# --- UPDATED LOADING AND RUNNING SECTION ---
try:
    # 1. Load the file
    df = pd.read_excel("need.xlsx")
    
    # 2. Let's see what the computer actually sees (Diagnostic)
    actual_col_count = len(df.columns)
    print(f"I found {actual_col_count} columns in your Excel file.")
    print(f"Column names found: {list(df.columns)}")

       # 3. Create a cleaned version of the data
    clean_df = pd.DataFrame()
   
   # Column A: Date
    clean_df['Date'] = pd.to_datetime(df.iloc[:, 0]) 
   
   # Column C: Hour (Converting "09:00" string to integer 9)
   # We convert to string, split at the colon, take the first part, and make it a number
    clean_df['Hour'] = pd.to_numeric(df.iloc[:, 2].astype(str).str.split(':').str[0], errors='coerce')
   
   # Column D: FTE_Brut (Ensuring it is a number, not text)
    clean_df['FTE_Brut'] = pd.to_numeric(df.iloc[:, 4], errors='coerce').fillna(0)

   # Clean up any rows that aren't valid numbers
    clean_df = clean_df.dropna(subset=['Hour'])


    
    print(f"Successfully processed {len(clean_df)} rows of data.")

    
    all_res = []
    for d in clean_df['Date'].unique():
        # Convert numpy datetime back to pandas timestamp for the solver
        ts = pd.to_datetime(d)
        day_data = clean_df[clean_df['Date'] == d]
        
        # Map Hour to FTE
        demand_dict = dict(zip(day_data['Hour'], day_data['FTE_Brut']))
        
        
        #---DEBUG PRINT----
        print(f"Total Demand for {ts.date()}:{sum(demand_dict.values())}")
        print(f"Hours with demand:{list(demand_dict.keys())}")
    
        # Solve for the day
        all_res.extend(solve_day(ts, demand_dict))

    print (demand_dict)
        
    if len(all_res) > 0:
        output_df = pd.DataFrame(all_res)
        output_df.to_excel("final_schedule.xlsx", index=False)
        print("\n*** SUCCESS! Check final_schedule.xlsx ***")
    else:
        print("\n*** ERROR: No assignments were generated. Check the text above for warnings. ***")

except Exception as e:
    # This will tell us EXACTLY what went wrong
    import traceback
    print("\n*** CRITICAL ERROR DETAILS ***")
    print(traceback.format_exc())

I found 15 columns in your Excel file.
Column names found: ['Date', 'Day', 'Hour', 'Volume', 'Sizing FTEs', 'Calls/Day/FTE', 'Cases/Day/FTE', 'Service Level %', 'Avg Wait Time (s)', 'Occupancy %', 'P(Wait) %', 'Immediate Answer %', 'Shift', 'Weekend', 'Unnamed: 14']
Successfully processed 2208 rows of data.
Total Demand for 2026-05-01:82.06153846153848
Hours with demand:[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23]
--- Day: 2026-05-01 (Weekday 4) | Agents Available: 50 ---


In [1]:
import pulp
import pandas as pd
from datetime import datetime

# --- 1. CONFIGURATION: YOUR AGENT LIST --- # Check these workdays! 0=Mon, 4=Fri, 5=Sat, 6=Sun.
agents_db = {}
for i in range(1, 51):
    if i <= 20: # 1-20: Morning (Mon-Fri)
        agents_db[f"Agent_{i:02d}"] = {"type": "FT", "workdays": [0,1,2,3,4], "window": (0, 9)}
    elif i <= 35: # 21-35: Afternoon (Mon-Fri)
        agents_db[f"Agent_{i:02d}"] = {"type": "FT", "workdays": [0,1,2,3,4], "window": (9, 23)}
    else: # 36-50: Everyone else (Full Week)
        agents_db[f"Agent_{i:02d}"] = {"type": "PT", "workdays": [0,1,2,3,4,5,6], "window": (0, 23)}

def solve_day(date_val, demand_dict):
    weekday = date_val.weekday()
    avail = [a for a, info in agents_db.items() if weekday in info['workdays']]
    
    # DEBUG: See how many agents are available
    print(f"--- Day: {date_val.date()} (Weekday {weekday}) | Agents Available: {len(avail)} ---")
    
    if len(avail) == 0:
        print("!!! WARNING: No agents available for this day of the week.")
        return []

    prob = pulp.LpProblem(f"Sch_{date_val.date()}", pulp.LpMinimize)
    slack_vars = {}
    choices = {}
    for a in avail:
        s_min, s_max = agents_db[a]['window']
        for s in range(s_min, s_max + 1):
            choices[(a, s)] = pulp.LpVariable(f"S_{a}_{s}_{date_val.date()}", cat="Binary")

    # Constraint: Max 1 shift
    for a in avail:
        s_min, s_max = agents_db[a]['window']
        prob += pulp.lpSum([choices[(a, s)] for s in range(s_min, s_max + 1)]) <= 1

    # Constraint: Coverage with "Slack" (allows it to finish even if understaffed)
    for h in range(24):
        coverage = []
        for a in avail:
            dur = 8 if agents_db[a]['type'] == "FT" else 4
            s_min, s_max = agents_db[a]['window']
            for s in range(s_min, s_max + 1):
                if s <= h < s + dur:
                    if agents_db[a]['type'] == "FT" and h == (s + 4): # Lunch
                        coverage.append(choices[(a, s)] * 0)
                    else: coverage.append(choices[(a, s)] * 1)
        
        # This 'slack' variable makes the code 'Optimal' even if you don't have enough staff
        slack = pulp.LpVariable(f"Slack_{h}", lowBound=0)
        slack_vars[h]=slack
        prob += pulp.lpSum(coverage) + slack >= demand_dict.get(h, 0)

   # Goal: Minimize agents used AND heavily penalize understaffing (Slack)
   # We multiply slack by 100 to make it "expensive" for the computer to ignore demand
    prob += pulp.lpSum([choices[v] for v in choices]) + pulp.lpSum([slack_vars[h] * 100 for h in range(24)])
    prob.solve(pulp.PULP_CBC_CMD(msg=1,timeLimit=30))
    
    res = []
    print(f"Status: {pulp.LpStatus[prob.status]}") # Should say 'Optimal'
    
    for (a, s) in choices:
        if pulp.value(choices[(a, s)]) == 1:
            dur = 8 if agents_db[a]['type'] == "FT" else 4
            res.append({"Date": date_val.date(), "Agent": a, "Start": f"{s:02d}:00", "End": f"{(s+dur)%24:02d}:00"})
    
    print(f"Assignments made: {len(res)}")
    return res

# --- UPDATED LOADING AND RUNNING SECTION ---
try:
    # 1. Load the file
    df = pd.read_excel("need.xlsx")
    
    # 2. Let's see what the computer actually sees (Diagnostic)
    actual_col_count = len(df.columns)
    print(f"I found {actual_col_count} columns in your Excel file.")
    print(f"Column names found: {list(df.columns)}")

       # 3. Create a cleaned version of the data
    clean_df = pd.DataFrame()
   
   # Column A: Date
    clean_df['Date'] = pd.to_datetime(df.iloc[:, 0]) 
   
   # Column C: Hour (Converting "09:00" string to integer 9)
   # We convert to string, split at the colon, take the first part, and make it a number
    clean_df['Hour'] = pd.to_numeric(df.iloc[:, 2].astype(str).str.split(':').str[0], errors='coerce')
   
   # Column D: FTE_Brut (Ensuring it is a number, not text)
    clean_df['FTE_Brut'] = pd.to_numeric(df.iloc[:, 4], errors='coerce').fillna(0)

   # Clean up any rows that aren't valid numbers
    clean_df = clean_df.dropna(subset=['Hour'])


    
    print(f"Successfully processed {len(clean_df)} rows of data.")

    
    all_res = []
    for d in clean_df['Date'].unique():
        # Convert numpy datetime back to pandas timestamp for the solver
        ts = pd.to_datetime(d)
        day_data = clean_df[clean_df['Date'] == d]
        
        # Map Hour to FTE
        demand_dict = dict(zip(day_data['Hour'], day_data['FTE_Brut']))
        
        
        #---DEBUG PRINT----
        print(f"Total Demand for {ts.date()}:{sum(demand_dict.values())}")
        print(f"Hours with demand:{list(demand_dict.keys())}")
    
        # Solve for the day
        all_res.extend(solve_day(ts, demand_dict))

    print (demand_dict)
        
    if len(all_res) > 0:
        output_df = pd.DataFrame(all_res)
        output_df.to_excel("final_schedule.xlsx", index=False)
        print("\n*** SUCCESS! Check final_schedule.xlsx ***")
    else:
        print("\n*** ERROR: No assignments were generated. Check the text above for warnings. ***")

except Exception as e:
    # This will tell us EXACTLY what went wrong
    import traceback
    print("\n*** CRITICAL ERROR DETAILS ***")
    print(traceback.format_exc())

I found 15 columns in your Excel file.
Column names found: ['Date', 'Day', 'Hour', 'Volume', 'Sizing FTEs', 'Calls/Day/FTE', 'Cases/Day/FTE', 'Service Level %', 'Avg Wait Time (s)', 'Occupancy %', 'P(Wait) %', 'Immediate Answer %', 'Shift', 'Weekend', 'Unnamed: 14']
Successfully processed 2208 rows of data.
Total Demand for 2026-05-01:82.06153846153848
Hours with demand:[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23]
--- Day: 2026-05-01 (Weekday 4) | Agents Available: 50 ---
Status: Optimal
Assignments made: 16
Total Demand for 2026-05-02:61.10769230769231
Hours with demand:[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23]
--- Day: 2026-05-02 (Weekday 5) | Agents Available: 15 ---
Status: Optimal
Assignments made: 15
Total Demand for 2026-05-03:61.90769230769231
Hours with demand:[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23]
--- Day: 2026-05-03 (Weekday 6) | Agents Avail

Status: Optimal
Assignments made: 16
Total Demand for 2026-06-03:78.76923076923079
Hours with demand:[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23]
--- Day: 2026-06-03 (Weekday 2) | Agents Available: 50 ---
Status: Optimal
Assignments made: 16
Total Demand for 2026-06-04:77.13846153846156
Hours with demand:[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23]
--- Day: 2026-06-04 (Weekday 3) | Agents Available: 50 ---
Status: Optimal
Assignments made: 16
Total Demand for 2026-06-05:73.35384615384616
Hours with demand:[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23]
--- Day: 2026-06-05 (Weekday 4) | Agents Available: 50 ---
Status: Optimal
Assignments made: 16
Total Demand for 2026-06-06:56.12307692307693
Hours with demand:[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23]
--- Day: 2026-06-06 (Weekday 5) | Agents Available: 15 ---
Status: Opti

Status: Optimal
Assignments made: 18
Total Demand for 2026-07-07:87.3846153846154
Hours with demand:[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23]
--- Day: 2026-07-07 (Weekday 1) | Agents Available: 50 ---
Status: Optimal
Assignments made: 17
Total Demand for 2026-07-08:95.23076923076921
Hours with demand:[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23]
--- Day: 2026-07-08 (Weekday 2) | Agents Available: 50 ---
Status: Optimal
Assignments made: 18
Total Demand for 2026-07-09:93.56923076923074
Hours with demand:[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23]
--- Day: 2026-07-09 (Weekday 3) | Agents Available: 50 ---
Status: Optimal
Assignments made: 18
Total Demand for 2026-07-10:87.3846153846154
Hours with demand:[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23]
--- Day: 2026-07-10 (Weekday 4) | Agents Available: 50 ---
Status: Optima